In [1]:
"""
LLaMA 3 Implementation for LULC Event Extraction - Complete Jupyter Notebook Version with Dates and JSON Output
========================================================================================================

This notebook demonstrates how to use LLaMA 3 for extracting Land Use Land Cover (LULC) 
change events from text, including temporal information and structured JSON output.

Requirements:
- Python 3.8+
- PyTorch 2.0+
- transformers 4.30.0+
- GPU with at least 16GB VRAM (for 8B model)
"""

# Cell 1: Import libraries and configure logging
import json
import os
import re
import logging
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Cell 2: Configuration settings
# You can modify these settings as needed
NER_OUTPUT_PATH = "extracted_entities_structured.json"
JSON_OUTPUT_PATH = "llama3_extracted_lulc_events.json"
CSV_OUTPUT_PATH = "llama3_extracted_lulc_events.csv"
SUMMARY_OUTPUT_PATH = "extraction_summary.json"

MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # 8B version
# Alternative models:
# "meta-llama/Meta-Llama-3-8B-Instruct" - Instruction-tuned version
# "meta-llama/Meta-Llama-3-70B" - Larger model (requires more VRAM)

NUM_SAMPLES = 10  # Processing first 10 documents for testing
USE_QUANTIZATION = True  # Set to True to use 4-bit quantization (reduces memory usage)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Configuration set:")
print(f"- Model: {MODEL_ID}")
print(f"- Device: {DEVICE}")
print(f"- Processing first {NUM_SAMPLES} documents")
print(f"- Using quantization: {USE_QUANTIZATION}")

Configuration set:
- Model: meta-llama/Meta-Llama-3-8B
- Device: cuda
- Processing first 10 documents
- Using quantization: True


In [3]:
# Cell 3: Setup model and tokenizer
def setup_model(model_id, device, use_quantization=False):
    """
    Load the LLaMA 3 model and tokenizer.
    
    Args:
        model_id: Hugging Face model ID
        device: Device to load the model on
        use_quantization: Whether to use 4-bit quantization
        
    Returns:
        model, tokenizer
    """
    logging.info(f"Loading LLaMA 3 tokenizer: {model_id}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Add padding token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Configure quantization if requested
    if use_quantization:
        logging.info("Using 4-bit quantization")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    else:
        quantization_config = None
    
    # Load model with appropriate configuration
    logging.info(f"Loading LLaMA 3 model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto" if device == "cuda" else None,
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        trust_remote_code=True
    )
    
    logging.info(f"LLaMA 3 model loaded successfully on {device}")
    return model, tokenizer

# Execute this cell to load the model
try:
    model, tokenizer = setup_model(MODEL_ID, DEVICE, USE_QUANTIZATION)
    print("✅ Model and tokenizer loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model, tokenizer = None, None

2025-05-23 13:44:28,629 - INFO - Loading LLaMA 3 tokenizer: meta-llama/Meta-Llama-3-8B
2025-05-23 13:44:29,273 - INFO - Using 4-bit quantization
2025-05-23 13:44:29,277 - INFO - Loading LLaMA 3 model: meta-llama/Meta-Llama-3-8B
2025-05-23 13:44:34,401 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-05-23 13:44:42,617 - INFO - LLaMA 3 model loaded successfully on cuda


✅ Model and tokenizer loaded successfully!


In [4]:
# Cell 4: Date extraction functions
def extract_dates_from_entities(entities):
    """
    Extract date information from entities.
    
    Args:
        entities: List of entity dictionaries
        
    Returns:
        Dictionary with date information
    """
    dates = []
    date_ranges = []
    
    # Extract DATE entities
    date_entities = [ent for ent in entities if ent.get('label') == 'DATE']
    
    for entity in date_entities:
        date_text = entity.get('text', '')
        if date_text:
            dates.append(date_text)
    
    # Try to identify date ranges
    if len(dates) >= 2:
        # Sort dates and create range
        dates_sorted = sorted(dates)
        date_ranges.append(f"{dates_sorted[0]} to {dates_sorted[-1]}")
    
    return {
        'dates': dates,
        'date_range': date_ranges[0] if date_ranges else "",
        'start_date': dates[0] if dates else "",
        'end_date': dates[-1] if len(dates) > 1 else ""
    }

def format_entities_for_prompt_with_dates(entities):
    """Enhanced version that formats entities for prompt inclusion."""
    if not entities:
        return "No specific entities pre-identified."
    
    # Group entities by type
    entities_by_type = {}
    for entity in entities:
        entity_type = entity.get('label', 'UNKNOWN')
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity.get('text', 'N/A'))
    
    # Format grouped entities
    formatted_lines = []
    for entity_type, entity_texts in entities_by_type.items():
        unique_texts = list(set(entity_texts))  # Remove duplicates
        entity_list = ", ".join([f'"{text}"' for text in unique_texts])
        formatted_lines.append(f"- {entity_type}: {entity_list}")
    
    return "\n".join(formatted_lines)

print("✅ Date extraction functions defined!")

✅ Date extraction functions defined!


In [5]:
# Cell 5: Enhanced prompt construction with dates
def construct_llama3_prompt_with_dates(sentence_text, entities):
    """
    Enhanced prompt construction that includes date extraction.
    """
    formatted_entities = format_entities_for_prompt_with_dates(entities)
    
    # Extract date information
    date_info = extract_dates_from_entities(entities)
    dates_context = f"Dates identified: {', '.join(date_info['dates'])}" if date_info['dates'] else "No dates identified"
    
    # Extract LULC entities specifically for better context
    lulc_entities = [ent for ent in entities if ent.get('label') == 'LULC']
    lulc_text = ", ".join([f'"{ent.get("text")}"' for ent in lulc_entities]) if lulc_entities else "None identified"
    
    # Extract change indicators for better context
    change_entities = [ent for ent in entities if ent.get('label') == 'CHANGE']
    change_text = ", ".join([f'"{ent.get("text")}"' for ent in change_entities]) if change_entities else "None identified"
    
    prompt = f"""<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
- Temporal information (DATE_RANGE, START_DATE, END_DATE)
</context>

<input>
"{sentence_text}"
</input>

<entities>
{formatted_entities}
</entities>

<date_context>
{dates_context}
</date_context>

<instructions>
Analyze the input text and extract a LULC change event with these components:

FROM: The original land use/cover type that is changing or being converted
TO: The resulting land use/cover type
CHANGE: Words indicating change (e.g., increase, decrease, conversion)
PROCESS: The broader process (e.g., deforestation, urbanization)
MAGNITUDE: Any percentage or area measurement
DATE_RANGE: The time period over which the change occurred
START_DATE: The beginning date/year of the change
END_DATE: The ending date/year of the change

If no LULC change event is present, respond with "NO_EVENT".
</instructions>

<examples>
Example 1:
Input: "Forest cover declined by 15% in the region between 2010 and 2020."
Entities:
- LULC: "Forest"
- CHANGE: "declined"
- PERCENT: "15%"
- LOC: "region"
- DATE: "2010", "2020"

Output:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%
DATE_RANGE: 2010 to 2020
START_DATE: 2010
END_DATE: 2020

Example 2:
Input: "Agricultural land was converted to urban areas in 2015."
Entities:
- LULC: "Agricultural land", "urban areas"
- CHANGE: "converted"
- DATE: "2015"

Output:
FROM: agricultural land
TO: urban areas
CHANGE: converted
PROCESS: urbanization
MAGNITUDE: 
DATE_RANGE: 2015
START_DATE: 2015
END_DATE: 

Example 3:
Input: "Built-up area increased from 52.88% in 2002 to 65.5% in 2018, a change of 12.77%."
Entities:
- LULC: "Built-up area"
- CHANGE: "increased"
- PERCENT: "52.88%", "65.5%", "12.77%"
- DATE: "2002", "2018"

Output:
FROM: built-up area
TO: built-up area
CHANGE: increased
PROCESS: urbanization
MAGNITUDE: 12.77%
DATE_RANGE: 2002 to 2018
START_DATE: 2002
END_DATE: 2018

Example 4:
Input: "The study examined biodiversity in tropical forests."
Entities:
- LULC: "tropical forests"

Output:
NO_EVENT
</examples>

<output>
"""
    return prompt

print("✅ Enhanced prompt construction function defined!")

✅ Enhanced prompt construction function defined!


In [6]:
# Cell 6: Text generation and parsing functions
def generate_with_llama3(model, tokenizer, prompt, max_new_tokens=256):
    """
    Generate text using LLaMA 3 model.
    
    Args:
        model: LLaMA 3 model
        tokenizer: LLaMA 3 tokenizer
        prompt: Input prompt
        max_new_tokens: Maximum number of tokens to generate
        
    Returns:
        Generated text
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4000).to(model.device)
    
    # Generate with appropriate parameters for structured extraction
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,  # Low temperature for deterministic outputs
            top_p=0.9,
            do_sample=True,  # Light sampling for better quality
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_output[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):]
    
    # Clean up the output
    generated_text = generated_text.strip()
    
    # If the output contains </output> tag, extract only the content before it
    if "</output>" in generated_text:
        generated_text = generated_text.split("</output>")[0].strip()
    
    return generated_text

def parse_llama3_output_with_dates(output_text):
    """
    Enhanced parsing that includes date information.
    
    Args:
        output_text: Raw output from LLaMA 3
        
    Returns:
        Dictionary with parsed fields including dates
    """
    # Check for NO_EVENT marker
    if "NO_EVENT" in output_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": "",
            "date_range": "",
            "start_date": "",
            "end_date": ""
        }
    
    # Extract fields using regex
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', output_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', output_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', output_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', output_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\nDATE_RANGE:|$)', output_text, re.DOTALL)
    date_range_match = re.search(r'DATE_RANGE:\s*(.*?)(?=\nSTART_DATE:|$)', output_text, re.DOTALL)
    start_date_match = re.search(r'START_DATE:\s*(.*?)(?=\nEND_DATE:|$)', output_text, re.DOTALL)
    end_date_match = re.search(r'END_DATE:\s*(.*?)(?=\n|$)', output_text, re.DOTALL)
    
    # Extract values or default to empty string
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    date_range = date_range_match.group(1).strip() if date_range_match else ""
    start_date = start_date_match.group(1).strip() if start_date_match else ""
    end_date = end_date_match.group(1).strip() if end_date_match else ""
    
    # Determine if an event was found (at least one field has content)
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude,
        "date_range": date_range,
        "start_date": start_date,
        "end_date": end_date
    }

def process_magnitude(magnitude):
    """
    Process magnitude into percent and area components.
    
    Args:
        magnitude: Raw magnitude string
        
    Returns:
        Tuple of (magnitude_percent, magnitude_area)
    """
    if not magnitude:
        return "", ""
    
    magnitude_percent = ""
    magnitude_area = ""
    
    # Check for percentage
    if "%" in magnitude:
        magnitude_percent = magnitude
    # Check for area units
    elif any(unit in magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
        magnitude_area = magnitude
    # Check for numbers with area units using regex
    elif re.search(r'\d+\s*(?:ha|km|m|acre)', magnitude, re.IGNORECASE):
        magnitude_area = magnitude
    # If it's just a number, try to determine if it's a percentage
    elif re.search(r'\d+\.\d+|\d+', magnitude):
        try:
            value = float(re.search(r'\d+\.\d+|\d+', magnitude).group())
            if value <= 100:
                magnitude_percent = magnitude
            else:
                magnitude_area = magnitude
        except:
            magnitude_area = magnitude
    else:
        magnitude_area = magnitude
    
    return magnitude_percent, magnitude_area

print("✅ Text generation and parsing functions defined!")

✅ Text generation and parsing functions defined!


In [7]:
# Cell 7: Load NER output data
# Execute this cell to load your data
try:
    logging.info(f"Loading NER output from {NER_OUTPUT_PATH}")
    with open(NER_OUTPUT_PATH, 'r', encoding='utf-8') as f:
        sentences_with_entities = json.load(f)
    
    total_sentences = len(sentences_with_entities)
    logging.info(f"Loaded {total_sentences} sentences with entities")
    
    # Limit to first NUM_SAMPLES for testing
    if NUM_SAMPLES > 0 and NUM_SAMPLES < total_sentences:
        sentences_with_entities = sentences_with_entities[:NUM_SAMPLES]
        logging.info(f"Processing first {NUM_SAMPLES} sentences for testing")
    else:
        logging.info(f"Processing all {total_sentences} sentences")
        
    # Display first example
    print(f"\n✅ Data loaded successfully!")
    print(f"📊 Total sentences available: {total_sentences}")
    print(f"🔧 Processing: {len(sentences_with_entities)} sentences")
    
    print(f"\n📝 First example:")
    if sentences_with_entities:
        example = sentences_with_entities[0]
        print(f"Sentence: {example.get('original_sentence', '')[:200]}...")
        print("Entities:")
        for entity in example.get('entities', [])[:5]:  # Show first 5 entities
            print(f"  - {entity.get('text', '')} ({entity.get('label', '')})")
        if len(example.get('entities', [])) > 5:
            print(f"  ... and {len(example.get('entities', [])) - 5} more entities")
        
except Exception as e:
    logging.error(f"Error loading NER output: {e}")
    sentences_with_entities = []
    print(f"❌ Error loading data: {e}")

2025-05-23 13:45:35,924 - INFO - Loading NER output from extracted_entities_structured.json
2025-05-23 13:45:35,957 - INFO - Loaded 1986 sentences with entities
2025-05-23 13:45:35,960 - INFO - Processing first 10 sentences for testing



✅ Data loaded successfully!
📊 Total sentences available: 1986
🔧 Processing: 10 sentences

📝 First example:
Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050....
Entities:
  - results (CHANGE)
  - Thimphu (LOC)
  - city (LULC)
  - changed (CHANGE)
  - change (CHANGE)
  ... and 1 more entities


In [8]:
# Cell 8: Test single example with dates
# Execute this cell to test extraction on a single example
if sentences_with_entities and model and tokenizer:
    print("🧪 Testing extraction on single example...")
    
    # Get the first example
    example = sentences_with_entities[0]
    sentence_text = example.get('original_sentence', '')
    entities = example.get('entities', [])
    
    print(f"\n📝 Input sentence:")
    print(f"'{sentence_text}'")
    
    # Extract date information
    date_info = extract_dates_from_entities(entities)
    print(f"\n📅 Date information extracted:")
    print(json.dumps(date_info, indent=2))
    
    # Construct prompt with dates
    prompt = construct_llama3_prompt_with_dates(sentence_text, entities)
    print(f"\n📋 Prompt length: {len(prompt)} characters")
    
    try:
        # Generate with LLaMA 3
        print(f"\n🤖 Generating response with LLaMA 3...")
        generated_text = generate_with_llama3(model, tokenizer, prompt)
        print(f"\n📤 Generated text:")
        print(generated_text)
        
        # Parse output with dates
        parsed_result = parse_llama3_output_with_dates(generated_text)
        print(f"\n📊 Parsed result (JSON format):")
        print(json.dumps(parsed_result, indent=2))
        
        # Process magnitude
        if parsed_result['magnitude']:
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            print(f"\n📏 Magnitude breakdown:")
            print(f"  magnitude_percent: {magnitude_percent}")
            print(f"  magnitude_area: {magnitude_area}")
            
        print("✅ Single example test completed successfully!")
        
    except Exception as e:
        print(f"❌ Error in single example test: {e}")
        
else:
    print("⚠️ Cannot run test - missing data, model, or tokenizer")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🧪 Testing extraction on single example...

📝 Input sentence:
'Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.'

📅 Date information extracted:
{
  "dates": [
    "2050"
  ],
  "date_range": "",
  "start_date": "2050",
  "end_date": ""
}

📋 Prompt length: 2663 characters

🤖 Generating response with LLaMA 3...

📤 Generated text:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%
DATE_RANGE: 2010 to 2020
START_DATE: 2010
END_DATE: 2020

📊 Parsed result (JSON format):
{
  "event_found": true,
  "from_lulc": "forest",
  "to_lulc": "CHANGE: declined\nPROCESS: deforestation\nMAGNITUDE: 15%\nDATE_RANGE: 2010 to 2020\nSTART_DATE: 2010\nEND_DATE: 2020",
  "change_indicator": "declined",
  "lulc_process": "deforestation",
  "magnitude": "15%",
  "date_range": "2010 to 2020",
  "start_date": "2010",
  "end_date": "2020"
}

📏 Magnitude breakdown:
  magnitude_pe

In [13]:
# Cell 9: Process all examples with JSON output
def process_all_examples_with_json_output():
    """Process all examples and return structured JSON results."""
    extracted_events = []
    
    if not model or not tokenizer:
        print("❌ Model or tokenizer not loaded!")
        return []
    
    print(f"🚀 Starting LULC event extraction with LLaMA 3...")
    print(f"📊 Processing {len(sentences_with_entities)} sentences")
    
    for idx, entry in enumerate(tqdm(sentences_with_entities, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Extract date information from entities
        date_info = extract_dates_from_entities(entities)
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'sentence_index': idx,
            'original_sentence': sentence_text,
            'extracted_entities': entities,
            'date_context': date_info,
            'llm_raw_output': 'Not Generated Yet',
            'extracted_event': {
                'event_found': False,
                'from_lulc': "",
                'to_lulc': "",
                'change_indicator': "",
                'lulc_process': "",
                'magnitude_percent': "",
                'magnitude_area': "",
                'date_range': "",
                'start_date': "",
                'end_date': ""
            },
            'processing_metadata': {
                'timestamp': datetime.now().isoformat(),
                'model_used': MODEL_ID,
                'sentence_length': len(sentence_text),
                'num_entities': len(entities),
                'error': None
            }
        }
        
        try:
            # Construct prompt with dates
            prompt = construct_llama3_prompt_with_dates(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output with dates
            parsed_result = parse_llama3_output_with_dates(generated_text)
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            
            # Update extracted_event
            event_data['extracted_event'] = {
                'event_found': parsed_result['event_found'],
                'from_lulc': parsed_result['from_lulc'],
                'to_lulc': parsed_result['to_lulc'],
                'change_indicator': parsed_result['change_indicator'],
                'lulc_process': parsed_result['lulc_process'],
                'magnitude_percent': magnitude_percent,
                'magnitude_area': magnitude_area,
                'date_range': parsed_result['date_range'],
                'start_date': parsed_result['start_date'],
                'end_date': parsed_result['end_date']
            }
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['processing_metadata']['error'] = str(e)
            event_data['llm_raw_output'] = f'Error during generation: {str(e)}'
        
        extracted_events.append(event_data)
    
    return extracted_events

# Execute processing
if sentences_with_entities and model and tokenizer:
    print("🎬 Starting extraction process...")
    all_extracted_events = process_all_examples_with_json_output()
    print(f"✅ Completed processing {len(all_extracted_events)} entries")
else:
    print("⚠️ Cannot start processing - missing data, model, or tokenizer")
    all_extracted_events = []

✅ Label Studio JSON format converter defined!
📊 Functions available:
- convert_to_labelstudio_format() - Main conversion function
- save_labelstudio_batch() - Save multiple tasks
- test_labelstudio_conversion() - Test with sample data
- get_labelstudio_config() - Get XML configuration


In [10]:
# Cell 10: Save outputs in both JSON and CSV formats
if all_extracted_events:
    print("💾 Saving results...")
    
    

💾 Saving results...


In [11]:
import json
import os
import re
import logging
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# Configuration settings
NER_OUTPUT_PATH = "extracted_entities_structured.json"  # your NER output file
JSON_OUTPUT_PATH = "llama3_extracted_lulc_events.json"   # JSON output path for full output
CSV_OUTPUT_PATH = "llama3_extracted_lulc_events.csv"       # CSV output path for flattened data
SUMMARY_OUTPUT_PATH = "extraction_summary.json"            # Summary output

MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # model id you want to use
NUM_SAMPLES = 10  # Process first 10 docs
USE_QUANTIZATION = True  # Use 4-bit quantization if needed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#######################################
# Cell 3: Setup model and tokenizer
#######################################
def setup_model(model_id, device, use_quantization=False):
    """
    Load the LLaMA 3 model and tokenizer.
    
    Args:
        model_id: Hugging Face model ID
        device: Device to load the model on
        use_quantization: Whether to use 4-bit quantization
        
    Returns:
        model, tokenizer
    """
    logging.info(f"Loading LLaMA 3 tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    if use_quantization:
        logging.info("Using 4-bit quantization")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    else:
        quantization_config = None
    
    logging.info(f"Loading LLaMA 3 model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    
    logging.info(f"LLaMA 3 model loaded successfully on {device}")
    return model, tokenizer

model, tokenizer = setup_model(MODEL_ID, DEVICE, USE_QUANTIZATION)

#######################################
# Cell 4 & 5: Define helper functions (date extraction, prompt construction, generation, and parsing)
#######################################
def extract_dates_from_entities(entities):
    """
    Extract date information from entities.
    
    Args:
        entities: List of entity dictionaries
        
    Returns:
        Dictionary with date information.
    """
    dates = []
    date_entities = [ent for ent in entities if ent.get('label') == 'DATE']
    for entity in date_entities:
        date_text = entity.get('text', '')
        if date_text:
            dates.append(date_text)
    # Sort and create a date range if at least 2 dates exist
    date_range = f"{dates[0]} to {dates[-1]}" if len(dates) >= 2 else (dates[0] if dates else "")
    return {
        'dates': dates,
        'date_range': date_range,
        'start_date': dates[0] if dates else "",
        'end_date': dates[-1] if len(dates) > 1 else ""
    }

def format_entities_for_prompt_with_dates(entities):
    """Format entities for inclusion in the prompt (including date entities)."""
    if not entities:
        return "No specific entities pre-identified."
    
    entities_by_type = {}
    for entity in entities:
        entity_type = entity.get('label', 'UNKNOWN')
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity.get('text', 'N/A'))
    
    formatted_lines = []
    for entity_type, entity_texts in entities_by_type.items():
        unique_texts = list(set(entity_texts))
        entity_list = ", ".join(['"' + text + '"' for text in unique_texts])
        formatted_lines.append(f"- {entity_type}: {entity_list}")
    
    return "\n".join(formatted_lines)

def construct_llama3_prompt_with_dates(sentence_text, entities):
    """
    Enhanced prompt construction that includes date extraction.
    """
    formatted_entities = format_entities_for_prompt_with_dates(entities)
    date_info = extract_dates_from_entities(entities)
    dates_context = f"Dates identified: {', '.join(date_info['dates'])}" if date_info['dates'] else "No dates identified"
    
    # Extract LULC and CHANGE entities for better context
    lulc_entities = [ent for ent in entities if ent.get('label') == 'LULC']
    lulc_text = ", ".join(['"' + ent.get("text") + '"' for ent in lulc_entities]) if lulc_entities else "None identified"
    
    change_entities = [ent for ent in entities if ent.get('label') == 'CHANGE']
    change_text = ", ".join(['"' + ent.get("text") + '"' for ent in change_entities]) if change_entities else "None identified"
    
    prompt = f"""<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
- Temporal information (DATE_RANGE, START_DATE, END_DATE)
</context>

<input>
"{sentence_text}"
</input>

<entities>
{formatted_entities}
</entities>

<date_context>
{dates_context}
</date_context>

<instructions>
Analyze the input text and extract a LULC change event with these components:

FROM: The original land use/cover type that is changing or being converted
TO: The resulting land use/cover type
CHANGE: Words indicating change (e.g., increase, decrease, conversion)
PROCESS: The broader process (e.g., deforestation, urbanization)
MAGNITUDE: Any percentage or area measurement
DATE_RANGE: The time period over which the change occurred
START_DATE: The beginning date/year of the change
END_DATE: The ending date/year of the change

If no LULC change event is present, respond with "NO_EVENT".
</instructions>

<examples>
Example 1:
Input: "Forest cover declined by 15% in the region between 2010 and 2020."
Entities:
- LULC: "Forest"
- CHANGE: "declined"
- PERCENT: "15%"
- LOC: "region"
- DATE: "2010", "2020"

Output:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%
DATE_RANGE: 2010 to 2020
START_DATE: 2010
END_DATE: 2020

Example 2:
Input: "Agricultural land was converted to urban areas in 2015."
Entities:
- LULC: "Agricultural land", "urban areas"
- CHANGE: "converted"
- DATE: "2015"

Output:
FROM: agricultural land
TO: urban areas
CHANGE: converted
PROCESS: urbanization
MAGNITUDE: 
DATE_RANGE: 2015
START_DATE: 2015
END_DATE: 

Example 3:
Input: "Built-up area increased from 52.88% in 2002 to 65.5% in 2018, a change of 12.77%."
Entities:
- LULC: "Built-up area"
- CHANGE: "increased"
- PERCENT: "52.88%", "65.5%", "12.77%"
- DATE: "2002", "2018"

Output:
FROM: built-up area
TO: built-up area
CHANGE: increased
PROCESS: urbanization
MAGNITUDE: 12.77%
DATE_RANGE: 2002 to 2018
START_DATE: 2002
END_DATE: 2018
</examples>

<output>
"""
    return prompt

def generate_with_llama3(model, tokenizer, prompt, max_new_tokens=256):
    """
    Generate text using LLaMA 3 model.
    
    Args:
        model: LLaMA 3 model.
        tokenizer: LLaMA 3 tokenizer.
        prompt: Input prompt.
        max_new_tokens: Maximum number of tokens to generate.
        
    Returns:
        Generated text.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_decoded = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
    generated_text = full_output[len(prompt_decoded):].strip()
    if "</output>" in generated_text:
        generated_text = generated_text.split("</output>")[0].strip()
    return generated_text

def parse_llama3_output_with_dates(output_text):
    """
    Parse the LLaMA 3 output including date information.
    
    Args:
        output_text: Raw output from LLaMA 3.
        
    Returns:
        Dictionary with parsed fields.
    """
    if "NO_EVENT" in output_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": "",
            "date_range": "",
            "start_date": "",
            "end_date": ""
        }
    
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', output_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', output_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', output_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', output_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\nDATE_RANGE:|$)', output_text, re.DOTALL)
    date_range_match = re.search(r'DATE_RANGE:\s*(.*?)(?=\nSTART_DATE:|$)', output_text, re.DOTALL)
    start_date_match = re.search(r'START_DATE:\s*(.*?)(?=\nEND_DATE:|$)', output_text, re.DOTALL)
    end_date_match = re.search(r'END_DATE:\s*(.*?)(?=\n|$)', output_text, re.DOTALL)
    
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    date_range = date_range_match.group(1).strip() if date_range_match else ""
    start_date = start_date_match.group(1).strip() if start_date_match else ""
    end_date = end_date_match.group(1).strip() if end_date_match else ""
    
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude,
        "date_range": date_range,
        "start_date": start_date,
        "end_date": end_date
    }

def process_magnitude(magnitude):
    """
    Process magnitude string into percentage and area components.
    
    Args:
        magnitude: Raw magnitude string.
        
    Returns:
        Tuple of (magnitude_percent, magnitude_area).
    """
    if not magnitude:
        return "", ""
    magnitude_percent = ""
    magnitude_area = ""
    if "%" in magnitude:
        magnitude_percent = magnitude
    elif any(unit in magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
        magnitude_area = magnitude
    elif re.search(r'\d+\s*(?:ha|km|m|acre)', magnitude, re.IGNORECASE):
        magnitude_area = magnitude
    elif re.search(r'\d+\.\d+|\d+', magnitude):
        try:
            value = float(re.search(r'\d+\.\d+|\d+', magnitude).group())
            if value <= 100:
                magnitude_percent = magnitude
            else:
                magnitude_area = magnitude
        except:
            magnitude_area = magnitude
    else:
        magnitude_area = magnitude
    return magnitude_percent, magnitude_area

#######################################
# Cell 6: Load NER output data and limit to first 10 documents
#######################################
try:
    logging.info(f"Loading NER output from {NER_OUTPUT_PATH}")
    with open(NER_OUTPUT_PATH, 'r', encoding='utf-8') as f:
        sentences_with_entities = json.load(f)
    
    total_sentences = len(sentences_with_entities)
    logging.info(f"Loaded {total_sentences} sentences with entities")
    
    if NUM_SAMPLES > 0 and NUM_SAMPLES < total_sentences:
        sentences_with_entities = sentences_with_entities[:NUM_SAMPLES]
        logging.info(f"Processing first {NUM_SAMPLES} sentences")
    else:
        logging.info(f"Processing all {total_sentences} sentences")
        
    # Display first example
    print("\nFirst example:")
    print(f"Sentence: {sentences_with_entities[0].get('original_sentence', '')}")
    print("Entities:")
    for entity in sentences_with_entities[0].get('entities', []):
        print(f"  - {entity.get('text', '')} ({entity.get('label', '')})")
        
except Exception as e:
    logging.error(f"Error loading NER output: {e}")
    sentences_with_entities = []

#######################################
# Cell 7: Test extraction on a single example with dates
#######################################
if sentences_with_entities:
    example = sentences_with_entities[0]
    sentence_text = example.get('original_sentence', '')
    entities = example.get('entities', [])
    
    # Extract date information
    date_info = extract_dates_from_entities(entities)
    print("Date information extracted:")
    print(json.dumps(date_info, indent=2))
    
    # Construct prompt with dates
    prompt = construct_llama3_prompt_with_dates(sentence_text, entities)
    print("\nPrompt:")
    print(prompt)
    
    # Generate with LLaMA 3
    generated_text = generate_with_llama3(model, tokenizer, prompt)
    print("\nGenerated text:")
    print(generated_text)
    
    # Parse output with dates
    parsed_result = parse_llama3_output_with_dates(generated_text)
    print("\nParsed result (JSON format):")
    print(json.dumps(parsed_result, indent=2))
    
    if parsed_result['magnitude']:
        magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
        print("\nMagnitude breakdown:")
        print(f"  magnitude_percent: {magnitude_percent}")
        print(f"  magnitude_area: {magnitude_area}")

#######################################
# Cell 8: Process all examples and output in JSON and CSV formats
#######################################
def process_all_examples_with_json_output():
    extracted_events = []
    
    logging.info("Starting LULC event extraction with LLaMA 3 (with dates and JSON output)")
    
    for idx, entry in enumerate(tqdm(sentences_with_entities, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        date_info = extract_dates_from_entities(entities)
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'extracted_entities': entities,
            'date_context': date_info,
            'llm_raw_output': 'Not Generated Yet',
            'extracted_event': {
                'event_found': False,
                'from_lulc': "",
                'to_lulc': "",
                'change_indicator': "",
                'lulc_process': "",
                'magnitude_percent': "",
                'magnitude_area': "",
                'date_range': "",
                'start_date': "",
                'end_date': ""
            },
            'processing_metadata': {
                'timestamp': pd.Timestamp.now().isoformat(),
                'model_used': MODEL_ID,
                'error': None
            }
        }
        
        try:
            prompt = construct_llama3_prompt_with_dates(sentence_text, entities)
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            parsed_result = parse_llama3_output_with_dates(generated_text)
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            
            event_data['extracted_event'] = {
                'event_found': parsed_result['event_found'],
                'from_lulc': parsed_result['from_lulc'],
                'to_lulc': parsed_result['to_lulc'],
                'change_indicator': parsed_result['change_indicator'],
                'lulc_process': parsed_result['lulc_process'],
                'magnitude_percent': magnitude_percent,
                'magnitude_area': magnitude_area,
                'date_range': parsed_result['date_range'],
                'start_date': parsed_result['start_date'],
                'end_date': parsed_result['end_date']
            }
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['processing_metadata']['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
    
    return extracted_events

print("Starting extraction process...")
all_extracted_events = process_all_examples_with_json_output()
print(f"Completed processing {len(all_extracted_events)} entries")

#######################################
# Cell 9: Save outputs in both JSON and CSV formats
#######################################
print("Saving results in JSON format...")
with open(JSON_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_extracted_events, f, indent=2, ensure_ascii=False)
print(f"JSON results saved to: {JSON_OUTPUT_PATH}")

print("Creating CSV format...")
csv_data = []
for event in all_extracted_events:
    flat_row = {
        'article_id': event['article_id'],
        'original_sentence': event['original_sentence'],
        'llm_raw_output': event['llm_raw_output'],
        'event_found': event['extracted_event']['event_found'],
        'from_lulc': event['extracted_event']['from_lulc'],
        'to_lulc': event['extracted_event']['to_lulc'],
        'change_indicator': event['extracted_event']['change_indicator'],
        'lulc_process': event['extracted_event']['lulc_process'],
        'magnitude_percent': event['extracted_event']['magnitude_percent'],
        'magnitude_area': event['extracted_event']['magnitude_area'],
        'date_range': event['extracted_event']['date_range'],
        'start_date': event['extracted_event']['start_date'],
        'end_date': event['extracted_event']['end_date'],
        'num_entities': len(event['extracted_entities']),
        'processing_timestamp': event['processing_metadata']['timestamp'],
        'error': event['processing_metadata']['error']
    }
    csv_data.append(flat_row)

df = pd.DataFrame(csv_data)
df.to_csv(CSV_OUTPUT_PATH, index=False, encoding='utf-8')
print(f"CSV results saved to: {CSV_OUTPUT_PATH}")

events_found = sum(1 for event in all_extracted_events if event['extracted_event']['event_found'])
events_with_dates = sum(1 for event in all_extracted_events if event['extracted_event']['date_range'])
events_with_magnitude = sum(1 for event in all_extracted_events if 
                             event['extracted_event']['magnitude_percent'] or event['extracted_event']['magnitude_area'])

summary = {
    'extraction_summary': {
        'total_sentences_processed': len(all_extracted_events),
        'events_found': events_found,
        'events_with_dates': events_with_dates,
        'events_with_magnitude': events_with_magnitude,
        'success_rate': f"{(events_found/len(all_extracted_events)*100):.2f}%" if all_extracted_events else "0%"
    },
    'processing_info': {
        'model_used': MODEL_ID,
        'processing_timestamp': datetime.now().isoformat(),
        'quantization_used': USE_QUANTIZATION,
        'device_used': DEVICE
    },
    'sample_events': [
        event for event in all_extracted_events[:3] if event['extracted_event']['event_found']
    ]
}

with open(SUMMARY_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Summary saved to: {SUMMARY_OUTPUT_PATH}")
print("\nExtraction Summary:")
print(f"- Total sentences processed: {len(all_extracted_events)}")
print(f"- Events found: {events_found}")
print(f"- Events with dates: {events_with_dates}")
print(f"- Events with magnitude: {events_with_magnitude}")
print(f"- Success rate: {(events_found/len(all_extracted_events)*100):.2f}%" if all_extracted_events else "0%")

#######################################
# Cell 10: Display sample JSON output
#######################################
print("Sample JSON output structure:")
print("=" * 50)
if all_extracted_events:
    sample_event = None
    for event in all_extracted_events:
        if event['extracted_event']['event_found']:
            sample_event = event
            break
    if sample_event:
        print(json.dumps(sample_event, indent=2))
    else:
        print("No events found in the sample. Showing first entry:")
        print(json.dumps(all_extracted_events[0], indent=2))
else:
    print("No events processed yet. Run the processing cell first.")

2025-05-23 13:50:44,143 - INFO - Loading LLaMA 3 tokenizer: meta-llama/Meta-Llama-3-8B
2025-05-23 13:50:44,753 - INFO - Using 4-bit quantization
2025-05-23 13:50:44,757 - INFO - Loading LLaMA 3 model: meta-llama/Meta-Llama-3-8B


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2025-05-23 13:50:52,781 - INFO - LLaMA 3 model loaded successfully on cuda
2025-05-23 13:50:52,834 - INFO - Loading NER output from extracted_entities_structured.json
2025-05-23 13:50:52,860 - INFO - Loaded 1986 sentences with entities
2025-05-23 13:50:52,862 - INFO - Processing first 10 sentences



First example:
Sentence: Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.
Entities:
  - results (CHANGE)
  - Thimphu (LOC)
  - city (LULC)
  - changed (CHANGE)
  - change (CHANGE)
  - 2050 (DATE)
Date information extracted:
{
  "dates": [
    "2050"
  ],
  "date_range": "2050",
  "start_date": "2050",
  "end_date": ""
}

Prompt:
<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
- Temporal information (DATE_RANGE, START_DATE, END_DATE)
</context>

<input>
"Simulation results reveal that the landsc

2025-05-23 13:51:08,946 - INFO - Starting LULC event extraction with LLaMA 3 (with dates and JSON output)



Generated text:
FROM: forest
TO: 
CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%
DATE_RANGE: 2010 to 2020
START_DATE: 2010
END_DATE: 2020

Parsed result (JSON format):
{
  "event_found": true,
  "from_lulc": "forest",
  "to_lulc": "CHANGE: declined\nPROCESS: deforestation\nMAGNITUDE: 15%\nDATE_RANGE: 2010 to 2020\nSTART_DATE: 2010\nEND_DATE: 2020",
  "change_indicator": "declined",
  "lulc_process": "deforestation",
  "magnitude": "15%",
  "date_range": "2010 to 2020",
  "start_date": "2010",
  "end_date": "2020"
}

Magnitude breakdown:
  magnitude_percent: 15%
  magnitude_area: 
Starting extraction process...


Processing sentences:   0%|          | 0/10 [00:00<?, ?it/s]

Completed processing 10 entries
Saving results in JSON format...
JSON results saved to: llama3_extracted_lulc_events.json
Creating CSV format...
CSV results saved to: llama3_extracted_lulc_events.csv
Summary saved to: extraction_summary.json

Extraction Summary:
- Total sentences processed: 10
- Events found: 10
- Events with dates: 10
- Events with magnitude: 10
- Success rate: 100.00%
Sample JSON output structure:
{
  "article_id": "Article_1",
  "original_sentence": "Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.",
  "extracted_entities": [
    {
      "text": "results",
      "label": "CHANGE",
      "start_char": 11,
      "end_char": 18
    },
    {
      "text": "Thimphu",
      "label": "LOC",
      "start_char": 48,
      "end_char": 55
    },
    {
      "text": "city",
      "label": "LULC",
      "start_char": 56,
      "end_char": 60
    },
    {
      "t

In [14]:
# Cell 13 (Fixed): Label Studio Configuration and Relationship Mapping
def create_label_studio_config():
    """
    Create Label Studio configuration for entity relationship visualization.
    """
    config = """
    <View>
      <Text name="text" value="$text"/>
      
      <!-- Entity Labeling -->
      <Labels name="label" toName="text">
        <Label value="FROM_LULC" background="red" alias="From Land Use"/>
        <Label value="TO_LULC" background="blue" alias="To Land Use"/>
        <Label value="CHANGE" background="green" alias="Change Indicator"/>
        <Label value="PROCESS" background="orange" alias="LULC Process"/>
        <Label value="MAGNITUDE" background="purple" alias="Magnitude"/>
        <Label value="DATE_RANGE" background="brown" alias="Date Range"/>
        <Label value="START_DATE" background="pink" alias="Start Date"/>
        <Label value="END_DATE" background="cyan" alias="End Date"/>
        <Label value="LOCATION" background="yellow" alias="Location"/>
      </Labels>
      
      <!-- Relationship Labeling -->
      <Relations name="relation" toName="label">
        <Relation value="causes" background="red" alias="Causes"/>
        <Relation value="converts_to" background="blue" alias="Converts To"/>
        <Relation value="part_of" background="green" alias="Part Of"/>
        <Relation value="affects" background="orange" alias="Affects"/>
        <Relation value="occurs_during" background="purple" alias="Occurs During"/>
        <Relation value="occurs_at" background="brown" alias="Occurs At"/>
        <Relation value="measured_by" background="pink" alias="Measured By"/>
        <Relation value="temporal_relation" background="cyan" alias="Temporal Relation"/>
      </Relations>
    </View>
    """
    return config

def extract_entity_positions(text, entity_text):
    """
    Find start and end positions of entity text in the original sentence.
    """
    if not entity_text or not text:
        return None
    
    # Clean up entity text
    entity_clean = entity_text.strip().lower()
    text_lower = text.lower()
    
    # Try exact match first
    start = text_lower.find(entity_clean)
    if start != -1:
        return {"start": start, "end": start + len(entity_clean)}
    
    # Try partial matches for phrases
    words = entity_clean.split()
    for i in range(len(words)):
        for j in range(i + 1, len(words) + 1):
            phrase = " ".join(words[i:j])
            if len(phrase) > 2:
                start = text_lower.find(phrase)
                if start != -1:
                    return {"start": start, "end": start + len(phrase)}
    
    # If no match found, try individual meaningful words
    for word in words:
        if len(word) > 3:  # Only consider meaningful words
            start = text_lower.find(word)
            if start != -1:
                return {"start": start, "end": start + len(word)}
    
    return None

def create_label_studio_task_fixed(event_data, task_id):
    """
    Convert extracted event data to correct Label Studio format.
    """
    sentence = event_data['original_sentence']
    extracted_event = event_data['extracted_event']
    
    # Skip if no event found
    if not extracted_event.get('event_found', False):
        return None
    
    # Create annotations list
    annotations = []
    relations = []
    annotation_id = 0
    entity_id_map = {}  # Maps entity type to annotation ID
    
    # Define entity mappings from extracted event
    entity_mappings = [
        ('FROM_LULC', extracted_event.get('from_lulc', '')),
        ('TO_LULC', extracted_event.get('to_lulc', '')),
        ('CHANGE', extracted_event.get('change_indicator', '')),
        ('PROCESS', extracted_event.get('lulc_process', '')),
        ('MAGNITUDE', extracted_event.get('magnitude_percent', '') or extracted_event.get('magnitude_area', '')),
        ('DATE_RANGE', extracted_event.get('date_range', '')),
        ('START_DATE', extracted_event.get('start_date', '')),
        ('END_DATE', extracted_event.get('end_date', ''))
    ]
    
    # Extract entities with positions
    for entity_type, entity_text in entity_mappings:
        if entity_text and entity_text.strip():
            position = extract_entity_positions(sentence, entity_text)
            if position:
                annotation = {
                    "id": f"entity_{annotation_id}",
                    "type": "labels",
                    "value": {
                        "start": position["start"],
                        "end": position["end"],
                        "text": sentence[position["start"]:position["end"]],  # Use actual text from sentence
                        "labels": [entity_type]
                    },
                    "to_name": "text",
                    "from_name": "label"
                }
                annotations.append(annotation)
                entity_id_map[entity_type] = f"entity_{annotation_id}"
                annotation_id += 1
            else:
                # If position not found, create a dummy position at the end
                annotation = {
                    "id": f"entity_{annotation_id}",
                    "type": "labels",
                    "value": {
                        "start": len(sentence) - len(entity_text) if len(entity_text) < len(sentence) else 0,
                        "end": len(sentence),
                        "text": entity_text,
                        "labels": [entity_type]
                    },
                    "to_name": "text",
                    "from_name": "label"
                }
                annotations.append(annotation)
                entity_id_map[entity_type] = f"entity_{annotation_id}"
                annotation_id += 1
    
    # Create relationships based on LULC change logic
    relationship_rules = [
        # FROM_LULC relationships
        ('FROM_LULC', 'CHANGE', 'causes'),
        ('FROM_LULC', 'TO_LULC', 'converts_to'),
        ('FROM_LULC', 'PROCESS', 'part_of'),
        
        # CHANGE relationships
        ('CHANGE', 'PROCESS', 'part_of'),
        ('CHANGE', 'MAGNITUDE', 'affects'),
        ('CHANGE', 'DATE_RANGE', 'occurs_during'),
        
        # PROCESS relationships
        ('PROCESS', 'MAGNITUDE', 'causes'),
        ('PROCESS', 'TO_LULC', 'results_in'),
        
        # Temporal relationships
        ('START_DATE', 'END_DATE', 'temporal_relation'),
        ('START_DATE', 'DATE_RANGE', 'part_of'),
        ('END_DATE', 'DATE_RANGE', 'part_of'),
        ('DATE_RANGE', 'CHANGE', 'occurs_during'),
        ('DATE_RANGE', 'PROCESS', 'occurs_during'),
        
        # Magnitude relationships
        ('MAGNITUDE', 'CHANGE', 'measured_by'),
        ('MAGNITUDE', 'PROCESS', 'measured_by')
    ]
    
    # Create relation annotations
    relation_id = 0
    for from_entity, to_entity, relation_type in relationship_rules:
        if from_entity in entity_id_map and to_entity in entity_id_map:
            relation = {
                "id": f"relation_{relation_id}",
                "type": "relation",
                "to_name": "label",
                "from_name": "relation",
                "value": {
                    "from": entity_id_map[from_entity],
                    "to": entity_id_map[to_entity],
                    "type": relation_type
                }
            }
            relations.append(relation)
            relation_id += 1
    
    # Combine all annotations
    all_annotations = annotations + relations
    
    # Create the complete task in CORRECT Label Studio format
    task = {
        "data": {
            "text": sentence  # This is the required "text" key!
        },
        "annotations": [{
            "id": f"annotation_{task_id}",
            "created_username": "AI_Extractor",
            "created_ago": "0 minutes",
            "result": all_annotations
        }],
        "predictions": [],  # Optional predictions
        "id": task_id,
        "meta": {
            "article_id": event_data.get('article_id', ''),
            "sentence_index": event_data.get('sentence_index', 0),
            "extraction_confidence": "AI_Generated",
            "model_used": event_data.get('processing_metadata', {}).get('model_used', 'LLaMA-3'),
            "event_summary": {
                "from_lulc": extracted_event.get('from_lulc', ''),
                "to_lulc": extracted_event.get('to_lulc', ''),
                "process": extracted_event.get('lulc_process', ''),
                "magnitude": extracted_event.get('magnitude_percent', '') or extracted_event.get('magnitude_area', '')
            }
        }
    }
    
    return task

print("✅ Fixed Label Studio functions defined!")

✅ Fixed Label Studio functions defined!


In [15]:
# Cell 14 (Fixed): Convert extracted events to correct Label Studio format
def create_label_studio_dataset_fixed(extracted_events, output_filename="label_studio_lulc_relations_fixed.json"):
    """
    Convert all extracted events to correct Label Studio format.
    """
    label_studio_tasks = []
    task_id = 1
    
    print(f"🔄 Converting {len(extracted_events)} events to Label Studio format...")
    
    successful_conversions = 0
    skipped_no_events = 0
    errors = 0
    
    for event_data in tqdm(extracted_events, desc="Converting to Label Studio"):
        try:
            # Check if event was found
            if not event_data.get('extracted_event', {}).get('event_found', False):
                skipped_no_events += 1
                continue
                
            task = create_label_studio_task_fixed(event_data, task_id)
            if task:
                label_studio_tasks.append(task)
                successful_conversions += 1
                task_id += 1
        except Exception as e:
            logging.warning(f"Error converting task {task_id}: {e}")
            errors += 1
            continue
    
    # Save to JSON file
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(label_studio_tasks, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Label Studio dataset created:")
    print(f"- Total input events: {len(extracted_events)}")
    print(f"- Events with LULC changes: {successful_conversions}")
    print(f"- Skipped (no events): {skipped_no_events}")
    print(f"- Conversion errors: {errors}")
    print(f"- Final tasks created: {len(label_studio_tasks)}")
    print(f"- File saved as: {output_filename}")
    
    return label_studio_tasks, output_filename

# Convert extracted events if available
if 'all_extracted_events' in globals() and all_extracted_events:
    label_studio_tasks_fixed, ls_filename_fixed = create_label_studio_dataset_fixed(all_extracted_events)
    
    # Validate the first task structure
    if label_studio_tasks_fixed:
        print(f"\n🔍 Sample task structure validation:")
        sample_task = label_studio_tasks_fixed[0]
        print(f"✅ Has 'data' key: {'data' in sample_task}")
        print(f"✅ Has 'text' in data: {'text' in sample_task.get('data', {})}")
        print(f"✅ Has 'annotations' key: {'annotations' in sample_task}")
        print(f"✅ Text content: {sample_task['data']['text'][:100]}...")
        
else:
    print("⚠️ No extracted events available. Run the extraction cells first.")

🔄 Converting 10 events to Label Studio format...


Converting to Label Studio:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Label Studio dataset created:
- Total input events: 10
- Events with LULC changes: 10
- Skipped (no events): 0
- Conversion errors: 0
- Final tasks created: 10
- File saved as: label_studio_lulc_relations_fixed.json

🔍 Sample task structure validation:
✅ Has 'data' key: True
✅ Has 'text' in data: True
✅ Has 'annotations' key: True
✅ Text content: Simulation results reveal that the landscape of Thimphu city has changed considerably during the stu...


In [19]:
# Cell 1: Simple Relationship Extractor and Visualizer
import json
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import seaborn as sns
from collections import defaultdict, Counter

print("📦 Libraries imported for relationship visualization")


📦 Libraries imported for relationship visualization


In [20]:
# Cell 2: Extract relationships from your LLaMA output
def extract_relationships_from_llama_output(all_extracted_events):
    """
    Extract clean relationships from LLaMA 3 output without Label Studio complexity.
    """
    relationships = []
    entities = []
    
    for idx, event_data in enumerate(all_extracted_events):
        extracted_event = event_data.get('extracted_event', {})
        
        if not extracted_event.get('event_found', False):
            continue
            
        sentence = event_data.get('original_sentence', '')
        article_id = event_data.get('article_id', f'doc_{idx}')
        
        # Extract entities
        event_entities = {}
        entity_types = [
            ('FROM_LULC', extracted_event.get('from_lulc', '')),
            ('TO_LULC', extracted_event.get('to_lulc', '')),
            ('CHANGE', extracted_event.get('change_indicator', '')),
            ('PROCESS', extracted_event.get('lulc_process', '')),
            ('MAGNITUDE', extracted_event.get('magnitude_percent', '') or extracted_event.get('magnitude_area', '')),
            ('DATE_RANGE', extracted_event.get('date_range', '')),
            ('START_DATE', extracted_event.get('start_date', '')),
            ('END_DATE', extracted_event.get('end_date', ''))
        ]
        
        # Store entities
        for entity_type, entity_text in entity_types:
            if entity_text and entity_text.strip():
                entity_id = f"{article_id}_{entity_type}_{idx}"
                entity_record = {
                    'id': entity_id,
                    'type': entity_type,
                    'text': entity_text.strip(),
                    'sentence': sentence,
                    'article_id': article_id,
                    'event_id': idx
                }
                entities.append(entity_record)
                event_entities[entity_type] = entity_record
        
        # Define relationship rules
        relationship_rules = [
            ('FROM_LULC', 'CHANGE', 'causes'),
            ('FROM_LULC', 'TO_LULC', 'converts_to'),
            ('FROM_LULC', 'PROCESS', 'undergoes'),
            ('CHANGE', 'PROCESS', 'part_of'),
            ('CHANGE', 'MAGNITUDE', 'has_magnitude'),
            ('PROCESS', 'MAGNITUDE', 'measured_by'),
            ('CHANGE', 'DATE_RANGE', 'occurs_during'),
            ('PROCESS', 'DATE_RANGE', 'happens_in'),
            ('START_DATE', 'END_DATE', 'precedes'),
            ('START_DATE', 'DATE_RANGE', 'starts'),
            ('END_DATE', 'DATE_RANGE', 'ends')
        ]
        
        # Create relationships
        for from_type, to_type, relation_type in relationship_rules:
            if from_type in event_entities and to_type in event_entities:
                relationship = {
                    'from_entity': event_entities[from_type]['id'],
                    'from_type': from_type,
                    'from_text': event_entities[from_type]['text'],
                    'to_entity': event_entities[to_type]['id'],
                    'to_type': to_type,
                    'to_text': event_entities[to_type]['text'],
                    'relationship': relation_type,
                    'sentence': sentence,
                    'article_id': article_id,
                    'event_id': idx
                }
                relationships.append(relationship)
    
    return entities, relationships

# Extract relationships
if 'all_extracted_events' in globals() and all_extracted_events:
    entities, relationships = extract_relationships_from_llama_output(all_extracted_events)
    
    print(f"✅ Relationship extraction completed:")
    print(f"   📋 Entities found: {len(entities)}")
    print(f"   🔗 Relationships created: {len(relationships)}")
    print(f"   📊 Events processed: {sum(1 for e in all_extracted_events if e.get('extracted_event', {}).get('event_found', False))}")
else:
    print("⚠️ No extracted events available. Run the LLaMA extraction first.")
    entities, relationships = [], []

✅ Relationship extraction completed:
   📋 Entities found: 75
   🔗 Relationships created: 100
   📊 Events processed: 10


In [22]:
# Cell 3: Create Interactive Network Visualization with Plotly
def create_network_visualization(entities, relationships):
    """
    Create an interactive network graph using Plotly.
    """
    if not entities or not relationships:
        print("⚠️ No data to visualize")
        return None
    
    # Create NetworkX graph
    G = nx.Graph()
    
    # Add nodes
    entity_colors = {
        'FROM_LULC': 'red',
        'TO_LULC': 'blue', 
        'CHANGE': 'green',
        'PROCESS': 'orange',
        'MAGNITUDE': 'purple',
        'DATE_RANGE': 'brown',
        'START_DATE': 'pink',
        'END_DATE': 'cyan'
    }
    
    for entity in entities:
        G.add_node(entity['id'], 
                   label=entity['text'],
                   type=entity['type'],
                   color=entity_colors.get(entity['type'], 'gray'),
                   sentence=entity['sentence'][:100] + "...")
    
    # Add edges
    for rel in relationships:
        G.add_edge(rel['from_entity'], rel['to_entity'], 
                   relationship=rel['relationship'],
                   weight=1)
    
    # Create layout
    pos = nx.spring_layout(G, k=3, iterations=50)
    
    # Extract coordinates
    edge_x = []
    edge_y = []
    edge_info = []
    
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
        # Get relationship info
        rel_info = next((r['relationship'] for r in relationships 
                        if (r['from_entity'] == edge[0] and r['to_entity'] == edge[1]) or
                           (r['from_entity'] == edge[1] and r['to_entity'] == edge[0])), 'unknown')
        edge_info.append(rel_info)
    
    # Create edge trace
    edge_trace = go.Scatter(x=edge_x, y=edge_y,
                           line=dict(width=2, color='#888'),
                           hoverinfo='none',
                           mode='lines')
    
    # Create node traces by type
    node_traces = []
    
    for entity_type, color in entity_colors.items():
        # Get nodes of this type
        type_entities = [e for e in entities if e['type'] == entity_type]
        if not type_entities:
            continue
            
        node_x = [pos[entity['id']][0] for entity in type_entities]
        node_y = [pos[entity['id']][1] for entity in type_entities]
        node_text = [f"{entity['type']}: {entity['text']}" for entity in type_entities]
        node_hover = [f"Type: {entity['type']}<br>Text: {entity['text']}<br>Sentence: {entity['sentence']}" 
                     for entity in type_entities]
        
        node_trace = go.Scatter(x=node_x, y=node_y,
                               mode='markers+text',
                               hoverinfo='text',
                               hovertext=node_hover,
                               text=node_text,
                               textposition="middle center",
                               name=entity_type,
                               marker=dict(size=20,
                                         color=color,
                                         line=dict(width=2, color='white')))
        node_traces.append(node_trace)
    
    # Create figure
    fig = go.Figure(data=[edge_trace] + node_traces,
                   layout=go.Layout(
                       title='LULC Entity Relationship Network',
                       font_size=16,
                       showlegend=True,
                       hovermode='closest',
                       margin=dict(b=20,l=5,r=5,t=40),
                       annotations=[ dict(
                           text="Hover over nodes for details. Legend shows entity types.",
                           showarrow=False,
                           xref="paper", yref="paper",
                           x=0.005, y=-0.002 ) ],
                       xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                       yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                       width=1000,
                       height=800))
    
    # Save and show
    fig.write_html("lulc_relationship_network.html")
    fig.show()
    
    print(f"✅ Interactive network saved as: lulc_relationship_network.html")
    return fig

# Create visualization
if entities and relationships:
    network_fig = create_network_visualization(entities, relationships)
else:
    print("⚠️ No entities or relationships to visualize")

✅ Interactive network saved as: lulc_relationship_network.html


In [23]:
# Cell 4: Create Relationship Statistics Dashboard
def create_relationship_dashboard(entities, relationships):
    """
    Create comprehensive relationship analysis dashboard.
    """
    if not entities or not relationships:
        print("⚠️ No data for dashboard")
        return None
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Entity Type Distribution', 'Relationship Type Distribution',
                       'Relationship Frequency Matrix', 'Temporal Analysis'),
        specs=[[{"type": "pie"}, {"type": "bar"}],
               [{"type": "heatmap"}, {"type": "scatter"}]]
    )
    
    # 1. Entity type distribution
    entity_counts = Counter([e['type'] for e in entities])
    fig.add_trace(go.Pie(labels=list(entity_counts.keys()), 
                        values=list(entity_counts.values()),
                        name="Entities"), row=1, col=1)
    
    # 2. Relationship type distribution
    rel_counts = Counter([r['relationship'] for r in relationships])
    fig.add_trace(go.Bar(x=list(rel_counts.keys()), 
                        y=list(rel_counts.values()),
                        name="Relationships"), row=1, col=2)
    
    # 3. Relationship matrix
    from_types = [r['from_type'] for r in relationships]
    to_types = [r['to_type'] for r in relationships]
    
    # Create matrix
    all_types = list(set(from_types + to_types))
    matrix = []
    for from_type in all_types:
        row = []
        for to_type in all_types:
            count = sum(1 for r in relationships 
                       if r['from_type'] == from_type and r['to_type'] == to_type)
            row.append(count)
        matrix.append(row)
    
    fig.add_trace(go.Heatmap(z=matrix,
                            x=all_types,
                            y=all_types,
                            colorscale='Blues'), row=2, col=1)
    
    # 4. Temporal analysis (if date data available)
    date_entities = [e for e in entities if 'DATE' in e['type']]
    if date_entities:
        dates = [e['text'] for e in date_entities]
        date_counts = Counter(dates)
        fig.add_trace(go.Scatter(x=list(date_counts.keys()),
                                y=list(date_counts.values()),
                                mode='markers+lines',
                                name="Temporal"), row=2, col=2)
    
    # Update layout
    fig.update_layout(height=800, showlegend=False, title_text="LULC Relationship Analysis Dashboard")
    
    # Save and show
    fig.write_html("lulc_relationship_dashboard.html")
    fig.show()
    
    print(f"✅ Dashboard saved as: lulc_relationship_dashboard.html")
    return fig

# Create dashboard
if entities and relationships:
    dashboard_fig = create_relationship_dashboard(entities, relationships)

✅ Dashboard saved as: lulc_relationship_dashboard.html


In [24]:
# Cell 5: Export to Multiple Formats
def export_relationships_to_formats(entities, relationships):
    """
    Export relationships to multiple usable formats.
    """
    exports = {}
    
    # 1. CSV Format for Excel/Analysis
    df_entities = pd.DataFrame(entities)
    df_relationships = pd.DataFrame(relationships)
    
    df_entities.to_csv('lulc_entities.csv', index=False)
    df_relationships.to_csv('lulc_relationships.csv', index=False)
    
    exports['csv'] = ['lulc_entities.csv', 'lulc_relationships.csv']
    
    # 2. NetworkX GraphML for Gephi/Cytoscape
    G = nx.DiGraph()
    
    for entity in entities:
        G.add_node(entity['id'], 
                   label=entity['text'],
                   type=entity['type'],
                   sentence=entity['sentence'])
    
    for rel in relationships:
        G.add_edge(rel['from_entity'], rel['to_entity'],
                   relationship=rel['relationship'],
                   sentence=rel['sentence'])
    
    nx.write_graphml(G, "lulc_network.graphml")
    exports['graphml'] = 'lulc_network.graphml'
    
    # 3. Simple JSON for Custom Tools
    simple_json = {
        "nodes": [{"id": e['id'], "label": e['text'], "type": e['type']} for e in entities],
        "edges": [{"from": r['from_entity'], "to": r['to_entity'], "label": r['relationship']} 
                 for r in relationships],
        "summary": {
            "total_nodes": len(entities),
            "total_edges": len(relationships),
            "node_types": list(Counter([e['type'] for e in entities]).keys()),
            "relationship_types": list(Counter([r['relationship'] for r in relationships]).keys())
        }
    }
    
    with open('lulc_simple_network.json', 'w') as f:
        json.dump(simple_json, f, indent=2)
    
    exports['json'] = 'lulc_simple_network.json'
    
    # 4. Cytoscape.js format
    cytoscape_data = {
        "elements": {
            "nodes": [{"data": {"id": e['id'], "label": e['text'], "type": e['type']}} for e in entities],
            "edges": [{"data": {"id": f"edge_{i}", "source": r['from_entity'], "target": r['to_entity'], "label": r['relationship']}} 
                     for i, r in enumerate(relationships)]
        },
        "style": [
            {
                "selector": "node",
                "style": {"label": "data(label)", "text-valign": "center"}
            },
            {
                "selector": "edge", 
                "style": {"label": "data(label)", "curve-style": "bezier", "target-arrow-shape": "triangle"}
            }
        ]
    }
    
    with open('lulc_cytoscape.json', 'w') as f:
        json.dump(cytoscape_data, f, indent=2)
    
    exports['cytoscape'] = 'lulc_cytoscape.json'
    
    # 5. D3.js format
    d3_data = {
        "nodes": [{"id": e['id'], "name": e['text'], "group": e['type']} for e in entities],
        "links": [{"source": r['from_entity'], "target": r['to_entity'], "value": 1, "type": r['relationship']} 
                 for r in relationships]
    }
    
    with open('lulc_d3.json', 'w') as f:
        json.dump(d3_data, f, indent=2)
    
    exports['d3'] = 'lulc_d3.json'
    
    return exports

# Export to all formats
if entities and relationships:
    exports = export_relationships_to_formats(entities, relationships)
    
    print(f"✅ Data exported to multiple formats:")
    for format_name, files in exports.items():
        if isinstance(files, list):
            for file in files:
                print(f"   📄 {format_name.upper()}: {file}")
        else:
            print(f"   📄 {format_name.upper()}: {files}")
else:
    print("⚠️ No data to export")

✅ Data exported to multiple formats:
   📄 CSV: lulc_entities.csv
   📄 CSV: lulc_relationships.csv
   📄 GRAPHML: lulc_network.graphml
   📄 JSON: lulc_simple_network.json
   📄 CYTOSCAPE: lulc_cytoscape.json
   📄 D3: lulc_d3.json


In [26]:
# Cell 6 (Fixed): Create Simple HTML Visualization Page
def create_html_visualization_page():
    """
    Create a simple HTML page with embedded visualizations.
    """
    html_content = """<!DOCTYPE html>
<html>
<head>
    <title>LULC Relationship Visualization</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body { 
            font-family: Arial, sans-serif; 
            margin: 20px; 
            background-color: #f5f5f5;
        }
        .container { 
            max-width: 1200px; 
            margin: 0 auto; 
            background-color: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }
        .section { 
            margin-bottom: 40px; 
            border: 1px solid #ddd; 
            padding: 20px; 
            border-radius: 5px; 
            background-color: #fafafa;
        }
        .viz { 
            width: 100%; 
            height: 500px; 
            border: 1px solid #ccc; 
            border-radius: 5px;
        }
        h1, h2 { 
            color: #333; 
            text-align: center;
        }
        .download-links { 
            margin: 10px 0; 
            text-align: center;
        }
        .download-links a { 
            margin-right: 15px; 
            padding: 8px 16px; 
            background: #007bff; 
            color: white; 
            text-decoration: none; 
            border-radius: 5px;
            display: inline-block;
            margin-bottom: 5px;
        }
        .download-links a:hover {
            background: #0056b3;
        }
        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
            margin: 20px 0;
        }
        .stat-card {
            background: #e7f3ff;
            padding: 15px;
            border-radius: 8px;
            text-align: center;
            border-left: 4px solid #007bff;
        }
        .stat-number {
            font-size: 2em;
            font-weight: bold;
            color: #007bff;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }
        th, td {
            padding: 10px;
            text-align: left;
            border-bottom: 1px solid #ddd;
        }
        th {
            background-color: #007bff;
            color: white;
        }
        tr:nth-child(even) {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🌍 LULC Relationship Visualization Dashboard</h1>
        
        <div class="section">
            <h2>📊 Summary Statistics</h2>
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-number">{num_entities}</div>
                    <div>Total Entities</div>
                </div>
                <div class="stat-card">
                    <div class="stat-number">{num_relationships}</div>
                    <div>Total Relationships</div>
                </div>
                <div class="stat-card">
                    <div class="stat-number">{num_entity_types}</div>
                    <div>Entity Types</div>
                </div>
                <div class="stat-card">
                    <div class="stat-number">{num_rel_types}</div>
                    <div>Relationship Types</div>
                </div>
            </div>
            <p><strong>Entity Types:</strong> {entity_types}</p>
            <p><strong>Relationship Types:</strong> {relationship_types}</p>
        </div>
        
        <div class="section">
            <h2>📁 Download Data Files</h2>
            <div class="download-links">
                <a href="lulc_entities.csv">📋 Entities CSV</a>
                <a href="lulc_relationships.csv">🔗 Relationships CSV</a>
                <a href="lulc_network.graphml">🌐 GraphML (Gephi)</a>
                <a href="lulc_cytoscape.json">⚡ Cytoscape JSON</a>
                <a href="lulc_d3.json">📈 D3.js JSON</a>
                <a href="lulc_simple_network.json">📋 Simple JSON</a>
            </div>
        </div>
        
        <div class="section">
            <h2>🔗 Interactive Network</h2>
            <iframe src="lulc_relationship_network.html" class="viz"></iframe>
        </div>
        
        <div class="section">
            <h2>📈 Analysis Dashboard</h2>
            <iframe src="lulc_relationship_dashboard.html" class="viz"></iframe>
        </div>
        
        <div class="section">
            <h2>📋 Relationship Examples</h2>
            <div id="examples">Loading examples...</div>
        </div>
        
        <div class="section">
            <h2>🛠️ Tools for Further Analysis</h2>
            <ul>
                <li><strong>Gephi:</strong> Import lulc_network.graphml for advanced network analysis</li>
                <li><strong>Cytoscape:</strong> Use lulc_cytoscape.json for biological-style network visualization</li>
                <li><strong>Excel/R/Python:</strong> Use CSV files for statistical analysis</li>
                <li><strong>Web Development:</strong> Use D3.js JSON for custom web visualizations</li>
                <li><strong>NetworkX (Python):</strong> Load GraphML for programmatic analysis</li>
                <li><strong>Tableau:</strong> Import CSV files for business intelligence dashboards</li>
            </ul>
        </div>
        
        <div class="section">
            <h2>🔍 Quick Analysis</h2>
            <div id="quick-analysis">
                <p>Analysis will be loaded here...</p>
            </div>
        </div>
    </div>
    
    <script>
        // Load and display relationship examples
        function loadExamples() {{
            fetch('lulc_simple_network.json')
                .then(response => response.json())
                .then(data => {{
                    let examples = data.edges.slice(0, 15);  // First 15 relationships
                    let html = '<table><tr><th>From Entity</th><th>Relationship</th><th>To Entity</th><th>From Type</th><th>To Type</th></tr>';
                    
                    examples.forEach((edge, index) => {{
                        let fromNode = data.nodes.find(n => n.id === edge.from);
                        let toNode = data.nodes.find(n => n.id === edge.to);
                        html += `<tr>
                            <td>${{fromNode ? fromNode.label : edge.from}}</td>
                            <td><strong>${{edge.label}}</strong></td>
                            <td>${{toNode ? toNode.label : edge.to}}</td>
                            <td><span style="background-color:#e7f3ff; padding:2px 6px; border-radius:3px;">${{fromNode ? fromNode.type : 'Unknown'}}</span></td>
                            <td><span style="background-color:#fff3e7; padding:2px 6px; border-radius:3px;">${{toNode ? toNode.type : 'Unknown'}}</span></td>
                        </tr>`;
                    }});
                    
                    html += '</table>';
                    document.getElementById('examples').innerHTML = html;
                    
                    // Quick analysis
                    let analysis = `
                        <h3>Network Summary:</h3>
                        <ul>
                            <li><strong>Total Nodes:</strong> ${{data.summary.total_nodes}}</li>
                            <li><strong>Total Edges:</strong> ${{data.summary.total_edges}}</li>
                            <li><strong>Network Density:</strong> ${{(data.summary.total_edges / (data.summary.total_nodes * (data.summary.total_nodes - 1))).toFixed(4)}}</li>
                            <li><strong>Average Connections per Node:</strong> ${{(data.summary.total_edges * 2 / data.summary.total_nodes).toFixed(2)}}</li>
                        </ul>
                        <h3>Most Common Relationship Types:</h3>
                        <ul id="rel-types"></ul>
                    `;
                    
                    document.getElementById('quick-analysis').innerHTML = analysis;
                    
                    // Count relationship types
                    let relTypeCounts = {{}};
                    data.edges.forEach(edge => {{
                        relTypeCounts[edge.label] = (relTypeCounts[edge.label] || 0) + 1;
                    }});
                    
                    let sortedRelTypes = Object.entries(relTypeCounts)
                        .sort((a, b) => b[1] - a[1])
                        .slice(0, 5);
                    
                    let relTypesList = document.getElementById('rel-types');
                    sortedRelTypes.forEach(([type, count]) => {{
                        let li = document.createElement('li');
                        li.innerHTML = `<strong>${{type}}:</strong> ${{count}} relationships`;
                        relTypesList.appendChild(li);
                    }});
                }})
                .catch(error => {{
                    console.error('Error loading data:', error);
                    document.getElementById('examples').innerHTML = '<p style="color:red;">Could not load examples. Make sure lulc_simple_network.json exists in the same directory.</p>';
                    document.getElementById('quick-analysis').innerHTML = '<p style="color:red;">Could not load analysis data.</p>';
                }});
        }}
        
        // Load examples when page loads
        window.addEventListener('load', loadExamples);
    </script>
</body>
</html>"""
    
    # Fill in the template safely
    try:
        if entities and relationships:
            entity_types_list = list(set([e['type'] for e in entities]))
            relationship_types_list = list(set([r['relationship'] for r in relationships]))
            
            html_filled = html_content.format(
                num_entities=len(entities),
                num_relationships=len(relationships),
                num_entity_types=len(entity_types_list),
                num_rel_types=len(relationship_types_list),
                entity_types=", ".join(entity_types_list),
                relationship_types=", ".join(relationship_types_list)
            )
        else:
            html_filled = html_content.format(
                num_entities=0,
                num_relationships=0,
                num_entity_types=0,
                num_rel_types=0,
                entity_types="None",
                relationship_types="None"
            )
        
        with open('lulc_visualization_dashboard.html', 'w', encoding='utf-8') as f:
            f.write(html_filled)
        
        print(f"✅ Complete visualization dashboard created: lulc_visualization_dashboard.html")
        print(f"🌐 Open this file in your browser to see all visualizations!")
        
    except Exception as e:
        print(f"❌ Error creating HTML dashboard: {e}")
        print("Creating simple fallback version...")
        
        # Create simple fallback
        simple_html = f"""<!DOCTYPE html>
<html>
<head>
    <title>LULC Relationships</title>
</head>
<body>
    <h1>LULC Relationship Analysis</h1>
    <h2>Files Created:</h2>
    <ul>
        <li><a href="lulc_relationship_network.html">Interactive Network</a></li>
        <li><a href="lulc_relationship_dashboard.html">Analysis Dashboard</a></li>
        <li><a href="lulc_entities.csv">Entities CSV</a></li>
        <li><a href="lulc_relationships.csv">Relationships CSV</a></li>
    </ul>
    <h2>Statistics:</h2>
    <p>Entities: {len(entities) if entities else 0}</p>
    <p>Relationships: {len(relationships) if relationships else 0}</p>
</body>
</html>"""
        
        with open('lulc_simple_dashboard.html', 'w', encoding='utf-8') as f:
            f.write(simple_html)
        
        print(f"✅ Simple dashboard created: lulc_simple_dashboard.html")

# Create HTML dashboard
try:
    create_html_visualization_page()
except Exception as e:
    print(f"❌ Error in HTML creation: {e}")
    # Create minimal version
    if 'entities' in globals() and entities:
        print(f"📊 Data available: {len(entities)} entities, {len(relationships)} relationships")
    print("✅ Other visualization files should still be created successfully!")

❌ Error creating HTML dashboard: ' \n            font-family'
Creating simple fallback version...
✅ Simple dashboard created: lulc_simple_dashboard.html


In [27]:
# Cell 6.5: Alternative Simple Visualization (Backup)
def create_simple_text_analysis():
    """
    Create simple text-based analysis if HTML fails.
    """
    if not entities or not relationships:
        print("⚠️ No data available for analysis")
        return
    
    print("🔍 SIMPLE TEXT ANALYSIS")
    print("=" * 50)
    
    # Basic stats
    print(f"📊 BASIC STATISTICS:")
    print(f"   Total Entities: {len(entities)}")
    print(f"   Total Relationships: {len(relationships)}")
    print(f"   Unique Articles: {len(set([e.get('article_id', '') for e in entities]))}")
    
    # Entity types
    entity_types = {}
    for entity in entities:
        entity_type = entity.get('type', 'Unknown')
        if entity_type not in entity_types:
            entity_types[entity_type] = []
        entity_types[entity_type].append(entity.get('text', ''))
    
    print(f"\n🏷️ ENTITY TYPES:")
    for entity_type, texts in entity_types.items():
        unique_texts = list(set(texts))[:5]  # Show first 5 unique
        print(f"   {entity_type} ({len(texts)} total): {', '.join(unique_texts)}...")
    
    # Relationship types
    rel_types = {}
    for rel in relationships:
        rel_type = rel.get('relationship', 'Unknown')
        if rel_type not in rel_types:
            rel_types[rel_type] = 0
        rel_types[rel_type] += 1
    
    print(f"\n🔗 RELATIONSHIP TYPES:")
    for rel_type, count in sorted(rel_types.items(), key=lambda x: x[1], reverse=True):
        print(f"   {rel_type}: {count}")
    
    # Sample relationships
    print(f"\n📝 SAMPLE RELATIONSHIPS:")
    for i, rel in enumerate(relationships[:10]):
        from_text = rel.get('from_text', 'Unknown')[:20]
        to_text = rel.get('to_text', 'Unknown')[:20]
        rel_type = rel.get('relationship', 'Unknown')
        print(f"   {i+1}. '{from_text}' --{rel_type}--> '{to_text}'")
    
    # Save to text file
    with open('lulc_analysis.txt', 'w', encoding='utf-8') as f:
        f.write("LULC RELATIONSHIP ANALYSIS\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Total Entities: {len(entities)}\n")
        f.write(f"Total Relationships: {len(relationships)}\n\n")
        
        f.write("ENTITY TYPES:\n")
        for entity_type, texts in entity_types.items():
            f.write(f"{entity_type}: {len(texts)} entities\n")
        
        f.write("\nRELATIONSHIP TYPES:\n")
        for rel_type, count in rel_types.items():
            f.write(f"{rel_type}: {count} relationships\n")
        
        f.write("\nALL RELATIONSHIPS:\n")
        for rel in relationships:
            f.write(f"{rel.get('from_text', '')} --{rel.get('relationship', '')}--> {rel.get('to_text', '')}\n")
    
    print(f"\n✅ Text analysis saved to: lulc_analysis.txt")

# Create simple analysis
create_simple_text_analysis()

🔍 SIMPLE TEXT ANALYSIS
📊 BASIC STATISTICS:
   Total Entities: 75
   Total Relationships: 100
   Unique Articles: 1

🏷️ ENTITY TYPES:
   FROM_LULC (10 total): built-up area, forest, agricultural land...
   TO_LULC (10 total): built-up area, CHANGE: declined
PROCESS: deforestation
MAGNITUDE: 15%
DATE_RANGE: 2010 to 2020
START_DATE: 2010
END_DATE: 2020, urban areas...
   CHANGE (10 total): converted, declined, increased...
   PROCESS (10 total): deforestation, urbanization...
   MAGNITUDE (10 total): 12.77%, DATE_RANGE: 2015
START_DATE: 2015
END_DATE:, 15%...
   DATE_RANGE (10 total): 2002 to 2018, 2010 to 2020, 2015...
   START_DATE (10 total): 2010, 2015, 2002...
   END_DATE (5 total): 2018, 2020...

🔗 RELATIONSHIP TYPES:
   causes: 10
   converts_to: 10
   undergoes: 10
   part_of: 10
   has_magnitude: 10
   measured_by: 10
   occurs_during: 10
   happens_in: 10
   starts: 10
   precedes: 5
   ends: 5

📝 SAMPLE RELATIONSHIPS:
   1. 'forest' --causes--> 'declined'
   2. 'forest' --conve

In [28]:
# Cell 7 (Updated): Final summary without HTML dependency
def print_final_summary():
    """
    Print final summary of all created files and next steps.
    """
    print("🎉 RELATIONSHIP EXTRACTION AND VISUALIZATION COMPLETE!")
    print("=" * 60)
    
    if entities and relationships:
        print(f"✅ Successfully processed:")
        print(f"   📊 {len(entities)} entities extracted")
        print(f"   🔗 {len(relationships)} relationships created")
        print(f"   📄 Multiple output formats generated")
        
        print(f"\n📁 FILES CREATED:")
        files_status = [
            ("lulc_entities.csv", "Entity data for analysis"),
            ("lulc_relationships.csv", "Relationship data for analysis"),
            ("lulc_simple_network.json", "Simple network data"),
            ("lulc_network.graphml", "Gephi network file"),
            ("lulc_cytoscape.json", "Cytoscape visualization"),
            ("lulc_d3.json", "D3.js web visualization"),
            ("lulc_analysis.txt", "Text-based analysis"),
            ("lulc_relationship_network.html", "Interactive network"),
            ("lulc_relationship_dashboard.html", "Analysis dashboard")
        ]
        
        for filename, description in files_status:
            try:
                import os
                if os.path.exists(filename):
                    print(f"   ✅ {filename} - {description}")
                else:
                    print(f"   ⚠️ {filename} - Not created")
            except:
                print(f"   📄 {filename} - {description}")
        
        print(f"\n🎯 RECOMMENDED NEXT STEPS:")
        print("1. 📊 Open lulc_entities.csv and lulc_relationships.csv in Excel")
        print("2. 🌐 Open lulc_relationship_network.html in your browser")
        print("3. 📈 Open lulc_relationship_dashboard.html for statistics")
        print("4. 📋 Read lulc_analysis.txt for text summary")
        print("5. 🔧 Import lulc_network.graphml into Gephi for advanced analysis")
        
        print(f"\n💡 TOP RELATIONSHIPS FOUND:")
        rel_counts = {}
        for rel in relationships[:20]:  # Top 20
            rel_type = rel.get('relationship', 'Unknown')
            if rel_type not in rel_counts:
                rel_counts[rel_type] = 0
            rel_counts[rel_type] += 1
        
        for rel_type, count in sorted(rel_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"   🔗 {rel_type}: {count} instances")
        
        print(f"\n🌟 SUCCESS! Your LULC relationships are ready for analysis!")
        
    else:
        print("❌ No entities or relationships were found.")
        print("💡 Check your LLaMA 3 extraction results and try again.")
    
    print(f"\n🚀 NO LABEL STUDIO NEEDED - You have working visualizations!")

# Print final summary
print_final_summary()

🎉 RELATIONSHIP EXTRACTION AND VISUALIZATION COMPLETE!
✅ Successfully processed:
   📊 75 entities extracted
   🔗 100 relationships created
   📄 Multiple output formats generated

📁 FILES CREATED:
   ✅ lulc_entities.csv - Entity data for analysis
   ✅ lulc_relationships.csv - Relationship data for analysis
   ✅ lulc_simple_network.json - Simple network data
   ✅ lulc_network.graphml - Gephi network file
   ✅ lulc_cytoscape.json - Cytoscape visualization
   ✅ lulc_d3.json - D3.js web visualization
   ✅ lulc_analysis.txt - Text-based analysis
   ✅ lulc_relationship_network.html - Interactive network
   ✅ lulc_relationship_dashboard.html - Analysis dashboard

🎯 RECOMMENDED NEXT STEPS:
1. 📊 Open lulc_entities.csv and lulc_relationships.csv in Excel
2. 🌐 Open lulc_relationship_network.html in your browser
3. 📈 Open lulc_relationship_dashboard.html for statistics
4. 📋 Read lulc_analysis.txt for text summary
5. 🔧 Import lulc_network.graphml into Gephi for advanced analysis

💡 TOP RELATIONSHIPS 

In [2]:
# Cell 1: Single Sentence Test Setup
# Replace your extracted events with a single sentence for testing

# Create a sample single sentence event (you can replace this with your actual sentence)
single_sentence_event = {
    'article_id': 'test_article_1',
    'sentence_index': 0,
    'original_sentence': 'Forest cover declined by 15% in the region between 2010 and 2020 due to deforestation.',
    'extracted_event': {
        'event_found': True,
        'from_lulc': 'forest',
        'to_lulc': '',
        'change_indicator': 'declined',
        'lulc_process': 'deforestation',
        'magnitude_percent': '15%',
        'magnitude_area': '',
        'date_range': '2010 to 2020',
        'start_date': '2010',
        'end_date': '2020'
    },
    'processing_metadata': {
        'model_used': 'LLaMA-3',
        'timestamp': '2025-01-01T00:00:00',
        'num_entities': 5,
        'error': None
    }
}

# Create list with just this one event
single_sentence_events = [single_sentence_event]

print("✅ Single sentence test data created")
print(f"📝 Sentence: {single_sentence_event['original_sentence']}")
print(f"🔍 Event found: {single_sentence_event['extracted_event']['event_found']}")

✅ Single sentence test data created
📝 Sentence: Forest cover declined by 15% in the region between 2010 and 2020 due to deforestation.
🔍 Event found: True


In [3]:
# Cell 2 (Enhanced): Extract Relationships for Single Sentence
def extract_relationships_single_sentence(event_data):
    """
    Extract relationships from a single sentence event with more accurate targeting.
    """
    entities = []
    relationships = []
    extracted_event = event_data.get('extracted_event', {})
    
    if not extracted_event.get('event_found', False):
        print("❌ No event found in this sentence")
        return entities, relationships
    
    sentence = event_data.get('original_sentence', '')
    article_id = event_data.get('article_id', 'single_test')
    
    # 1. Extract entities with precise start and end positions
    entity_positions = {}  # Store {entity_type: (start, end)}
    
    entity_types = [
        ('FROM_LULC', extracted_event.get('from_lulc', '')),
        ('TO_LULC', extracted_event.get('to_lulc', '')),
        ('CHANGE', extracted_event.get('change_indicator', '')),
        ('PROCESS', extracted_event.get('lulc_process', '')),
        ('MAGNITUDE', extracted_event.get('magnitude_percent', '') or extracted_event.get('magnitude_area', '')),
        ('DATE_RANGE', extracted_event.get('date_range', '')),
        ('START_DATE', extracted_event.get('start_date', '')),
        ('END_DATE', extracted_event.get('end_date', ''))
    ]
    
    # 2. FIND the entities in the sentence
    for entity_type, entity_text in entity_types:
        if entity_text and entity_text.strip():
            start_pos = sentence.lower().find(entity_text.lower())  # Case-insensitive search
            if start_pos != -1:
                end_pos = start_pos + len(entity_text)
                entity_positions[entity_type] = (start_pos, end_pos)
                
                # Create and store entity info
                entity_id = f"{article_id}_{entity_type}_0"
                entity_record = {
                    'id': entity_id,
                    'type': entity_type,
                    'text': entity_text.strip(),
                    'sentence': sentence,
                    'article_id': article_id,
                    'event_id': 0,
                    'start': start_pos,
                    'end': end_pos
                }
                entities.append(entity_record)
                print(f"✅ FOUND: {entity_type}: '{entity_text}' at [{start_pos}:{end_pos}]")  # Debugging
            else:
                print(f"❌ NOT FOUND: {entity_type}: '{entity_text}'")
    
    
    # 3. Create Relationships (based on positions)
    relationship_rules = [
        ('FROM_LULC', 'CHANGE', 'causes'),
        ('FROM_LULC', 'TO_LULC', 'converts_to'),
        ('FROM_LULC', 'PROCESS', 'undergoes'),
        ('CHANGE', 'PROCESS', 'part_of'),
        ('CHANGE', 'MAGNITUDE', 'has_magnitude'),
        ('PROCESS', 'MAGNITUDE', 'measured_by'),
        ('CHANGE', 'DATE_RANGE', 'occurs_during'),
        ('PROCESS', 'DATE_RANGE', 'happens_in'),
        ('START_DATE', 'END_DATE', 'precedes'),
        ('START_DATE', 'DATE_RANGE', 'starts'),
        ('END_DATE', 'DATE_RANGE', 'ends')
    ]
    
    print("\n🔗 Creating relationships:")
    
    for from_type, to_type, relation_type in relationship_rules:
        from_entity = next((e for e in entities if e['type'] == from_type), None)
        to_entity = next((e for e in entities if e['type'] == to_type), None)
        
        if from_entity and to_entity:
            relationship = {
                'from_entity': from_entity['id'],
                'from_type': from_type,
                'from_text': from_entity['text'],
                'to_entity': to_entity['id'],
                'to_type': to_type,
                'to_text': to_entity['text'],
                'relationship': relation_type,
                'sentence': sentence,
                'article_id': article_id,
                'event_id': 0
            }
            relationships.append(relationship)
            print(f"   ✅ {from_type}('{from_entity['text']}') --{relation_type}--> {to_type}('{to_entity['text']}')")
    
    return entities, relationships

In [4]:
# Cell 3 (Enhanced): Proper Label Studio Format with Positions
def create_proper_label_studio_format_enhanced(sentence, entities):
    """
    Create Label Studio format with entity positions
    """
    label_studio_data = {
        "data": {
            "text": sentence
        },
        "annotations": [
            {
                "result": []
            }
        ]
    }

    result = []
    for entity in entities:
        # Add entity annotation using character positions
        entity_annotation = {
            "id": entity['id'],
            "type": "labels",
            "value": {
                "start": entity['start'],
                "end": entity['end'],
                "text": sentence[entity['start']:entity['end']],  # Extract original text
                "labels": [entity['type']]
            },
            "to_name": "text",
            "from_name": "label"
        }
        result.append(entity_annotation)

    label_studio_data["annotations"][0]["result"] = result
    return label_studio_data

In [6]:
# Cell 4 (Modified): Convert your Extracted Data
def convert_your_data_to_label_studio(all_extracted_events):
  label_studio_tasks = []

  for task_id, event in enumerate(all_extracted_events):
    # Extract the data we need
    sentence = event.get('original_sentence', '')
    extracted = event.get('extracted_event', {})

    task = {
        "data": {
            "text": sentence #The sentence
        },
        "id": task_id + 1,
        "annotations": [{
            "id": f"annotation_{task_id}",
            "created_username": "AI_Assistant",
            "created_ago": "1 minute",
            "result": [] #We add the results here
        }]
    }

    result = []

    entity_texts = [
                extracted.get('from_lulc', ''),
                extracted.get('to_lulc', ''),
                extracted.get('change_indicator', ''),
                extracted.get('lulc_process', ''),
                extracted.get('magnitude_percent', '') or extracted.get('magnitude_area', ''),
                extracted.get('date_range', ''),
                extracted.get('start_date', ''),
                extracted.get('end_date', '')
            ]

    entity_types = [
                'FROM_LULC',
                'TO_LULC',
                'CHANGE',
                'PROCESS',
                'MAGNITUDE',
                'DATE',
                'DATE',
                'DATE'
                ]

    entities_to_find = zip(entity_texts,entity_types) #list of tuples

    entity_id = 0
    entity_map = {} #a dictionary of Entity labels as keys and entity_id as value

    for entity_text, entity_type in entities_to_find:

            if entity_text and entity_text.strip():

                # Find the entity within the original text
                start = sentence.lower().find(entity_text.lower()) #Case insensitive
                if start!=-1: #Make sure we have something for the snippet later.
                    end = start + len(entity_text) #Find where the extracted entityText ends
                    entity_map[entity_type] = f"entity_{entity_id}" #add our entityLabel to the map
                    #Now create and add the Label-Studio annotation:
                    annotation = {
                        "id": f"entity_{entity_id}",
                        "type": "labels",
                        "value": {
                            "start": start, #Here we use the starting positions
                            "end": end,   #Here we use ending positions
                            "text": sentence[start:end], #Here we create the extracted text
                            "labels": [entity_type] #Here is our Label
                        },
                        "to_name": "text",
                        "from_name": "label"
                    }
                    task["annotations"][0]["result"].append(annotation) #Finally append
                    entity_id+=1

        # --------------------------------------------
        # ADD RELATIONSHIPS - Only if we have at least 2 entities
        # --------------------------------------------

        if (len(entity_map)>=2):

              relationshiptypes = [
                  ('FROM_LULC', 'CHANGE', 'causes'),
                  ('FROM_LULC', 'TO_LULC', 'converts_to'),
                  ('CHANGE', 'PROCESS', 'part_of'),
                  ('CHANGE', 'MAGNITUDE', 'affects'),
                  ('PROCESS', 'MAGNITUDE', 'causes')
                  ]

              for from_type, to_type, relation_type in relationshiptypes:

                  if (from_type in entity_map and to_type in entity_map):

                      source = entity_map[from_type] #Source!
                      target = entity_map[to_type]   #Target!

                      #Add the relationships
                      relation = {
                                  "id": f"relation_{entity_id}",
                                  "type": "relation",
                                  "to_name": "label",
                                  "from_name": "relation",
                                  "value": {
                                      "from": source, #<== Reference the Source
                                      "to": target, #<== Reference the Target
                                      "type": relation_type #Give the type
                                      }
                                  }
                      task["annotations"][0]["result"].append(relation)

                      entity_id+=1

    label_studio_tasks.append(task) #Append finally
  return label_studio_tasks

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 81)

In [7]:
# Cell 5: Create Pre-annotated Label Studio Data (Advanced)
def create_pre_annotated_label_studio(all_extracted_events):
    """
    Create Label Studio data WITH pre-annotations from your AI extraction.
    This will show your entities and relationships already marked!
    """
    if not all_extracted_events:
        print("❌ No data to pre-annotate!")
        return []
    
    pre_annotated_tasks = []
    
    for task_id, event in enumerate(all_extracted_events):
        if not event.get('extracted_event', {}).get('event_found', False):
            continue
            
        sentence = event.get('original_sentence', '')
        extracted = event.get('extracted_event', {})
        
        # Create basic task
        task = {
            "data": {
                "text": sentence
            },
            "annotations": [{
                "id": f"annotation_{task_id}",
                "created_username": "AI_Assistant",
                "created_ago": "1 minute",
                "result": []
            }],
            "id": task_id + 1
        }
        
        # Find entity positions and create annotations
        entity_id = 0
        entity_map = {}
        
        entities_to_find = [
            (extracted.get('from_lulc', ''), 'FROM_LULC'),
            (extracted.get('to_lulc', ''), 'TO_LULC'),
            (extracted.get('change_indicator', ''), 'CHANGE'),
            (extracted.get('lulc_process', ''), 'PROCESS'),
            (extracted.get('magnitude_percent', '') or extracted.get('magnitude_area', ''), 'MAGNITUDE'),
            (extracted.get('date_range', ''), 'DATE'),
            (extracted.get('start_date', ''), 'DATE'),
            (extracted.get('end_date', ''), 'DATE')
        ]
        
        for entity_text, entity_type in entities_to_find:
            if entity_text and entity_text.strip():
                # Try to find the entity in the sentence
                start_pos = sentence.lower().find(entity_text.lower())
                if start_pos != -1:
                    end_pos = start_pos + len(entity_text)
                    
                    annotation = {
                        "id": f"entity_{entity_id}",
                        "type": "labels",
                        "value": {
                            "start": start_pos,
                            "end": end_pos,
                            "text": entity_text,
                            "labels": [entity_type]
                        },
                        "to_name": "text",
                        "from_name": "label"
                    }
                    
                    task["annotations"][0]["result"].append(annotation)
                    entity_map[entity_type] = f"entity_{entity_id}"
                    entity_id += 1
        
        # Add relationships if we have entities
        if len(entity_map) >= 2:
            relationships = [
                ('FROM_LULC', 'CHANGE', 'causes'),
                ('FROM_LULC', 'TO_LULC', 'converts_to'),
                ('CHANGE', 'PROCESS', 'part_of'),
                ('CHANGE', 'MAGNITUDE', 'affects')
            ]
            
            for from_type, to_type, relation_type in relationships:
                if from_type in entity_map and to_type in entity_map:
                    relation = {
                        "id": f"relation_{len(task['annotations'][0]['result'])}",
                        "type": "relation",
                        "to_name": "label",
                        "from_name": "relation",
                        "value": {
                            "from": entity_map[from_type],
                            "to": entity_map[to_type],
                            "type": relation_type
                        }
                    }
                    task["annotations"][0]["result"].append(relation)
        
        pre_annotated_tasks.append(task)
    
    return pre_annotated_tasks

# Create pre-annotated version
if 'all_extracted_events' in globals() and all_extracted_events:
    pre_annotated = create_pre_annotated_label_studio(all_extracted_events)
    
    if pre_annotated:
        with open('pre_annotated_label_studio.json', 'w', encoding='utf-8') as f:
            json.dump(pre_annotated, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Pre-annotated Label Studio data created!")
        print(f"📊 File: pre_annotated_label_studio.json")
        print(f"📝 Contains {len(pre_annotated)} pre-annotated tasks")
        print(f"🎯 This will show your AI extractions already marked in Label Studio!")
    else:
        print("❌ Could not create pre-annotations!")

In [8]:
# Cell 6: Validation and Instructions
def validate_label_studio_json(filename):
    """
    Validate that the JSON will work with Label Studio.
    """
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        if not isinstance(data, list):
            print(f"❌ {filename}: Must be a list of tasks")
            return False
        
        for i, task in enumerate(data[:3]):  # Check first 3
            if 'data' not in task:
                print(f"❌ {filename}: Task {i} missing 'data' field")
                return False
            
            if 'text' not in task['data']:
                print(f"❌ {filename}: Task {i} missing 'text' in data")
                return False
            
            if not isinstance(task['data']['text'], str):
                print(f"❌ {filename}: Task {i} 'text' must be string")
                return False
        
        print(f"✅ {filename}: Valid Label Studio format!")
        print(f"📊 Contains {len(data)} tasks")
        return True
        
    except Exception as e:
        print(f"❌ {filename}: Error - {e}")
        return False

# Validate all created files
files_to_validate = [
    'proper_label_studio_import.json',
    'your_data_label_studio.json',
    'pre_annotated_label_studio.json'
]

print("🔧 VALIDATION RESULTS:")
print("=" * 40)

for filename in files_to_validate:
    try:
        import os
        if os.path.exists(filename):
            validate_label_studio_json(filename)
        else:
            print(f"⚠️ {filename}: File not found")
    except:
        pass

print(f"\n🎯 FINAL INSTRUCTIONS:")
print("=" * 40)
print("1. 🚀 Start Label Studio: label-studio start")
print("2. 🌐 Open: http://localhost:8080")
print("3. ➕ Create New Project")
print("4. ⚙️ Set up interface with: label_studio_config_simple.xml")
print("5. 📁 Import data:")
print("   - For simple annotation: proper_label_studio_import.json")
print("   - For your data: your_data_label_studio.json")
print("   - For pre-annotated: pre_annotated_label_studio.json")

print(f"\n📋 FILES CREATED:")
print("✅ proper_label_studio_import.json - Basic sentences for annotation")
print("✅ your_data_label_studio.json - Your extracted data")
print("✅ pre_annotated_label_studio.json - Pre-annotated with AI results")
print("✅ label_studio_config_simple.xml - Interface configuration")

print(f"\n🎉 These files WILL work with Label Studio!")

🔧 VALIDATION RESULTS:
✅ proper_label_studio_import.json: Valid Label Studio format!
📊 Contains 5 tasks
✅ your_data_label_studio.json: Valid Label Studio format!
📊 Contains 10 tasks
✅ pre_annotated_label_studio.json: Valid Label Studio format!
📊 Contains 10 tasks

🎯 FINAL INSTRUCTIONS:
1. 🚀 Start Label Studio: label-studio start
2. 🌐 Open: http://localhost:8080
3. ➕ Create New Project
4. ⚙️ Set up interface with: label_studio_config_simple.xml
5. 📁 Import data:
   - For simple annotation: proper_label_studio_import.json
   - For your data: your_data_label_studio.json
   - For pre-annotated: pre_annotated_label_studio.json

📋 FILES CREATED:
✅ proper_label_studio_import.json - Basic sentences for annotation
✅ your_data_label_studio.json - Your extracted data
✅ pre_annotated_label_studio.json - Pre-annotated with AI results
✅ label_studio_config_simple.xml - Interface configuration

🎉 These files WILL work with Label Studio!


In [9]:
# Cell 5: Create Pre-annotated Label Studio Data (Advanced)
def create_pre_annotated_label_studio(all_extracted_events):
    """
    Create Label Studio data WITH pre-annotations from your AI extraction.
    This will show your entities and relationships already marked!
    """
    if not all_extracted_events:
        print("❌ No data to pre-annotate!")
        return []
    
    pre_annotated_tasks = []
    
    for task_id, event in enumerate(all_extracted_events):
        if not event.get('extracted_event', {}).get('event_found', False):
            continue
            
        sentence = event.get('original_sentence', '')
        extracted = event.get('extracted_event', {})
        
        # Create basic task
        task = {
            "data": {
                "text": sentence
            },
            "annotations": [{
                "id": f"annotation_{task_id}",
                "created_username": "AI_Assistant",
                "created_ago": "1 minute",
                "result": []
            }],
            "id": task_id + 1
        }
        
        # Find entity positions and create annotations
        entity_id = 0
        entity_map = {}
        
        entities_to_find = [
            (extracted.get('from_lulc', ''), 'FROM_LULC'),
            (extracted.get('to_lulc', ''), 'TO_LULC'),
            (extracted.get('change_indicator', ''), 'CHANGE'),
            (extracted.get('lulc_process', ''), 'PROCESS'),
            (extracted.get('magnitude_percent', '') or extracted.get('magnitude_area', ''), 'MAGNITUDE'),
            (extracted.get('date_range', ''), 'DATE'),
            (extracted.get('start_date', ''), 'DATE'),
            (extracted.get('end_date', ''), 'DATE')
        ]
        
        for entity_text, entity_type in entities_to_find:
            if entity_text and entity_text.strip():
                # Try to find the entity in the sentence
                start_pos = sentence.lower().find(entity_text.lower())
                if start_pos != -1:
                    end_pos = start_pos + len(entity_text)
                    
                    annotation = {
                        "id": f"entity_{entity_id}",
                        "type": "labels",
                        "value": {
                            "start": start_pos,
                            "end": end_pos,
                            "text": entity_text,
                            "labels": [entity_type]
                        },
                        "to_name": "text",
                        "from_name": "label"
                    }
                    
                    task["annotations"][0]["result"].append(annotation)
                    entity_map[entity_type] = f"entity_{entity_id}"
                    entity_id += 1
        
        # Add relationships if we have entities
        if len(entity_map) >= 2:
            relationships = [
                ('FROM_LULC', 'CHANGE', 'causes'),
                ('FROM_LULC', 'TO_LULC', 'converts_to'),
                ('CHANGE', 'PROCESS', 'part_of'),
                ('CHANGE', 'MAGNITUDE', 'affects')
            ]
            
            for from_type, to_type, relation_type in relationships:
                if from_type in entity_map and to_type in entity_map:
                    relation = {
                        "id": f"relation_{len(task['annotations'][0]['result'])}",
                        "type": "relation",
                        "to_name": "label",
                        "from_name": "relation",
                        "value": {
                            "from": entity_map[from_type],
                            "to": entity_map[to_type],
                            "type": relation_type
                        }
                    }
                    task["annotations"][0]["result"].append(relation)
        
        pre_annotated_tasks.append(task)
    
    return pre_annotated_tasks

# Create pre-annotated version
if 'all_extracted_events' in globals() and all_extracted_events:
    pre_annotated = create_pre_annotated_label_studio(all_extracted_events)
    
    if pre_annotated:
        with open('pre_annotated_label_studio.json', 'w', encoding='utf-8') as f:
            json.dump(pre_annotated, f, indent=2, ensure_ascii=False)
        
        print(f"✅ Pre-annotated Label Studio data created!")
        print(f"📊 File: pre_annotated_label_studio.json")
        print(f"📝 Contains {len(pre_annotated)} pre-annotated tasks")
        print(f"🎯 This will show your AI extractions already marked in Label Studio!")
    else:
        print("❌ Could not create pre-annotations!")

In [10]:
# Cell 6: Validation and Instructions
def validate_label_studio_json(filename):
    """
    Validate that the JSON will work with Label Studio.
    """
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        if not isinstance(data, list):
            print(f"❌ {filename}: Must be a list of tasks")
            return False
        
        for i, task in enumerate(data[:3]):  # Check first 3
            if 'data' not in task:
                print(f"❌ {filename}: Task {i} missing 'data' field")
                return False
            
            if 'text' not in task['data']:
                print(f"❌ {filename}: Task {i} missing 'text' in data")
                return False
            
            if not isinstance(task['data']['text'], str):
                print(f"❌ {filename}: Task {i} 'text' must be string")
                return False
        
        print(f"✅ {filename}: Valid Label Studio format!")
        print(f"📊 Contains {len(data)} tasks")
        return True
        
    except Exception as e:
        print(f"❌ {filename}: Error - {e}")
        return False

# Validate all created files
files_to_validate = [
    'proper_label_studio_import.json',
    'your_data_label_studio.json',
    'pre_annotated_label_studio.json'
]

print("🔧 VALIDATION RESULTS:")
print("=" * 40)

for filename in files_to_validate:
    try:
        import os
        if os.path.exists(filename):
            validate_label_studio_json(filename)
        else:
            print(f"⚠️ {filename}: File not found")
    except:
        pass

print(f"\n🎯 FINAL INSTRUCTIONS:")
print("=" * 40)
print("1. 🚀 Start Label Studio: label-studio start")
print("2. 🌐 Open: http://localhost:8080")
print("3. ➕ Create New Project")
print("4. ⚙️ Set up interface with: label_studio_config_simple.xml")
print("5. 📁 Import data:")
print("   - For simple annotation: proper_label_studio_import.json")
print("   - For your data: your_data_label_studio.json")
print("   - For pre-annotated: pre_annotated_label_studio.json")

print(f"\n📋 FILES CREATED:")
print("✅ proper_label_studio_import.json - Basic sentences for annotation")
print("✅ your_data_label_studio.json - Your extracted data")
print("✅ pre_annotated_label_studio.json - Pre-annotated with AI results")
print("✅ label_studio_config_simple.xml - Interface configuration")

print(f"\n🎉 These files WILL work with Label Studio!")

🔧 VALIDATION RESULTS:
✅ proper_label_studio_import.json: Valid Label Studio format!
📊 Contains 5 tasks
✅ your_data_label_studio.json: Valid Label Studio format!
📊 Contains 10 tasks
✅ pre_annotated_label_studio.json: Valid Label Studio format!
📊 Contains 10 tasks

🎯 FINAL INSTRUCTIONS:
1. 🚀 Start Label Studio: label-studio start
2. 🌐 Open: http://localhost:8080
3. ➕ Create New Project
4. ⚙️ Set up interface with: label_studio_config_simple.xml
5. 📁 Import data:
   - For simple annotation: proper_label_studio_import.json
   - For your data: your_data_label_studio.json
   - For pre-annotated: pre_annotated_label_studio.json

📋 FILES CREATED:
✅ proper_label_studio_import.json - Basic sentences for annotation
✅ your_data_label_studio.json - Your extracted data
✅ pre_annotated_label_studio.json - Pre-annotated with AI results
✅ label_studio_config_simple.xml - Interface configuration

🎉 These files WILL work with Label Studio!


In [11]:
def create_single_sentence_entry(event, article_id, sentence_index):
    """
    Create a Label Studio task for one sentence with proper relations.
    Ensuring IDs match exactly what Label Studio expects.
    """
    ee = event.get("extracted_event", {})
    
    # Create nodes with correct ID format that Label Studio can use for relations
    nodes = {}
    
    # FROM_LULC node
    if ee.get("from_lulc"):
        nodes["FROM_LULC"] = {
            "id": f"FROM_LULC_{sentence_index}",  # Simplified ID format
            "text": ee.get("from_lulc"),
            "type": "FROM_LULC"
        }
    
    # TO_LULC node
    if ee.get("to_lulc"):
        nodes["TO_LULC"] = {
            "id": f"TO_LULC_{sentence_index}",
            "text": ee.get("to_lulc"),
            "type": "TO_LULC"
        }
    
    # CHANGE node
    if ee.get("change_indicator"):
        nodes["CHANGE"] = {
            "id": f"CHANGE_{sentence_index}",
            "text": ee.get("change_indicator"),
            "type": "CHANGE"
        }
    
    # PROCESS node
    if ee.get("lulc_process"):
        nodes["PROCESS"] = {
            "id": f"PROCESS_{sentence_index}",
            "text": ee.get("lulc_process"),
            "type": "PROCESS"
        }
    
    # MAGNITUDE node
    magnitude = ""
    if ee.get("magnitude_percent"):
        magnitude = ee.get("magnitude_percent")
    elif ee.get("magnitude_area"):
        magnitude = ee.get("magnitude_area")
    if magnitude:
        nodes["MAGNITUDE"] = {
            "id": f"MAGNITUDE_{sentence_index}",
            "text": magnitude,
            "type": "MAGNITUDE"
        }
    
    # DATE nodes if available
    if ee.get("date_range"):
        nodes["DATE_RANGE"] = {
            "id": f"DATE_RANGE_{sentence_index}",
            "text": ee.get("date_range"),
            "type": "DATE_RANGE"
        }
    
    # Create nodes list
    nodes_list = list(nodes.values())
    
    # KEY CHANGE: Build relations with the EXACT same ID format used in nodes
    relations = []
    
    # FROM_LULC causes CHANGE
    if "FROM_LULC" in nodes and "CHANGE" in nodes:
        relations.append({
            "from_id": nodes["FROM_LULC"]["id"],  # Use from_id instead of from
            "to_id": nodes["CHANGE"]["id"],       # Use to_id instead of to
            "type": "causes"                      # Use type instead of relation
        })
    
    # CHANGE part_of PROCESS
    if "CHANGE" in nodes and "PROCESS" in nodes:
        relations.append({
            "from_id": nodes["CHANGE"]["id"],
            "to_id": nodes["PROCESS"]["id"],
            "type": "part_of"
        })
    
    # FROM_LULC converts_to TO_LULC
    if "FROM_LULC" in nodes and "TO_LULC" in nodes:
        relations.append({
            "from_id": nodes["FROM_LULC"]["id"],
            "to_id": nodes["TO_LULC"]["id"],
            "type": "converts_to"
        })
    
    # CHANGE affects MAGNITUDE
    if "CHANGE" in nodes and "MAGNITUDE" in nodes:
        relations.append({
            "from_id": nodes["CHANGE"]["id"],
            "to_id": nodes["MAGNITUDE"]["id"],
            "type": "affects"
        })
    
    # PROCESS causes MAGNITUDE
    if "PROCESS" in nodes and "MAGNITUDE" in nodes:
        relations.append({
            "from_id": nodes["PROCESS"]["id"],
            "to_id": nodes["MAGNITUDE"]["id"],
            "type": "causes"
        })
    
    # Final task structure with correct format for Label Studio
    task_entry = {
        "data": {
            "text": event.get("original_sentence", ""),
            "relations": relations,  # Relations must match Label Studio's expected format
            "entities": nodes_list   # Use "entities" instead of "nodes" for Label Studio
        }
    }
    
    return task_entry

# Test on one sentence
event = all_extracted_events[0]  # Adjust with your actual data
sentence_index = 0
single_task = create_single_sentence_entry(event, "Article_1", sentence_index)

# Save to JSON
LABEL_STUDIO_JSON_PATH = "label_studio_relations.json"
with open(LABEL_STUDIO_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump([single_task], f, indent=2, ensure_ascii=False)  # Wrap in list for Label Studio

print(f"Label Studio JSON file saved to: {LABEL_STUDIO_JSON_PATH}")
print("Sample JSON structure:")
print(json.dumps(single_task, indent=2))

NameError: name 'all_extracted_events' is not defined

In [12]:
import json

def create_single_sentence_entry(event, article_id, sentence_index):
    """
    Create a Label Studio task for one sentence.
    Each node now has a "text" key (required by Label Studio) and an optional "type" field.
    """
    ee = event.get("extracted_event", {})

    # Create nodes if non-empty.
    nodes = {}
    if ee.get("from_lulc"):
        nodes["FROM_LULC"] = {
            "id": f"{article_id}_FROM_LULC_{sentence_index}",
            "text": ee.get("from_lulc"),
            "type": "FROM_LULC"
        }
    if ee.get("to_lulc"):
        nodes["TO_LULC"] = {
            "id": f"{article_id}_TO_LULC_{sentence_index}",
            "text": ee.get("to_lulc"),
            "type": "TO_LULC"
        }
    if ee.get("change_indicator"):
        nodes["CHANGE"] = {
            "id": f"{article_id}_CHANGE_{sentence_index}",
            "text": ee.get("change_indicator"),
            "type": "CHANGE"
        }
    if ee.get("lulc_process"):
        nodes["PROCESS"] = {
            "id": f"{article_id}_PROCESS_{sentence_index}",
            "text": ee.get("lulc_process"),
            "type": "PROCESS"
        }
    # For MAGNITUDE, combine (choose one that is available)
    magnitude = ""
    if ee.get("magnitude_percent"):
        magnitude = ee.get("magnitude_percent")
    elif ee.get("magnitude_area"):
        magnitude = ee.get("magnitude_area")
    if magnitude:
        nodes["MAGNITUDE"] = {
            "id": f"{article_id}_MAGNITUDE_{sentence_index}",
            "text": magnitude,
            "type": "MAGNITUDE"
        }
    # Optional: add DATE information if available
    if ee.get("date_range"):
        nodes["DATE_RANGE"] = {
            "id": f"{article_id}_DATE_RANGE_{sentence_index}",
            "text": ee.get("date_range"),
            "type": "DATE_RANGE"
        }
    if ee.get("start_date"):
        nodes["START_DATE"] = {
            "id": f"{article_id}_START_DATE_{sentence_index}",
            "text": ee.get("start_date"),
            "type": "START_DATE"
        }
    if ee.get("end_date"):
        nodes["END_DATE"] = {
            "id": f"{article_id}_END_DATE_{sentence_index}",
            "text": ee.get("end_date"),
            "type": "END_DATE"
        }
    
    nodes_list = list(nodes.values())

    # Build relations from available nodes; 
    # Adjust rules as needed.
    relations = []
    if "FROM_LULC" in nodes and "CHANGE" in nodes:
        relations.append({
            "from": nodes["FROM_LULC"]["id"],
            "to": nodes["CHANGE"]["id"],
            "relation": "causes"
        })
    if "CHANGE" in nodes and "PROCESS" in nodes:
        relations.append({
            "from": nodes["CHANGE"]["id"],
            "to": nodes["PROCESS"]["id"],
            "relation": "part_of"
        })
    if "FROM_LULC" in nodes and "TO_LULC" in nodes:
        relations.append({
            "from": nodes["FROM_LULC"]["id"],
            "to": nodes["TO_LULC"]["id"],
            "relation": "converts_to"
        })
    if "CHANGE" in nodes and "MAGNITUDE" in nodes:
        relations.append({
            "from": nodes["CHANGE"]["id"],
            "to": nodes["MAGNITUDE"]["id"],
            "relation": "affects"
        })
    if "PROCESS" in nodes and "MAGNITUDE" in nodes:
        relations.append({
            "from": nodes["PROCESS"]["id"],
            "to": nodes["MAGNITUDE"]["id"],
            "relation": "causes"
        })

    # Final task structure must include a "text" field.
    task_entry = {
        "data": {
            "event_id": f"{article_id}_event_{sentence_index}",
            "text": event.get("original_sentence", ""),
            "nodes": nodes_list,
            "relations": relations
        }
    }
    return task_entry

# Example: Assume 'event' is the first event from your extraction list.
# Replace with your actual event dictionary.
event = all_extracted_events[0]  # for a single sentence
article_id = "Article_1"         # change as needed
sentence_index = 0               # if you have multiple sentences per article

# Generate single sentence task entry.
single_sentence_entry = create_single_sentence_entry(event, article_id, sentence_index)

# For testing, print the JSON structure
print(json.dumps(single_sentence_entry, indent=2))

NameError: name 'all_extracted_events' is not defined